# GUI-Model: Mobile UI Transition & Action Predictor

8 Qwen-family Vision-Language 모델 × 3 학습 데이터셋 (`AndroidControl` / `AndroidControl_2` / `MonkeyCollection`) + `MobiBench` (평가 전용 벤치마크) 매트릭스를 uv 가 관리하는 단일 `.venv` + LlamaFactory 백엔드로 운영한다.

**Stage 1 / Stage 2 mode (`full` / `lora`)** 은 쉘 스크립트의 `--stage1-mode` · `--stage2-mode` 플래그로 선택한다 (Stage 1 default `full`, Stage 2 default `lora`). 두 모드의 학습 YAML 은 Section 0 에서 자동 생성되며, 산출물 경로 (`adapters/`, `merged/epoch-{E}/`, `eval/.../on-{EVAL_DS}/`) 와 HF Hub repo ID 는 모드별로 분리된다.

**실행 흐름 (Stage 1 / Stage 2 공통): `train → merge → eval`**. eval 은 HF Hub 에 push 된 merged repo 를 pull 해 sweep 하므로 학습 머신이 아닌 환경에서도 재실행 가능하다.


## Dataset Overview

본 프로젝트는 5 개 모바일 UI 인터랙션 데이터셋을 처리한다. 학습 대상 DS 는 **AndroidControl (AC)**, **AndroidControl_2 (AC_2)**, **AndroidControl_3 (AC_3)**, **MonkeyCollection (MC)** 네 가지이며 (단 MC 는 Stage 1 전용), **MobiBench (MB)** 는 OOD 일반화 측정용 평가 전용 벤치마크다.

| DS | 단축 코드 | LlamaFactory prefix | 학습 | Stage 2 | test 형태 | 비고 |
|---|---|---|---|---|---|---|
| AndroidControl | `AC` | `GUI-Model-AC` | ✓ | ✓ | ID/OOD 2 파일 (app-level) | 주력 DS, `episodes_meta.jsonl` 기반 split |
| AndroidControl_2 | `AC_2` | `GUI-Model-AC_2` | ✓ | ✓ | 단일 test (사전 분할) | AC images 공유, token budget 5400 override |
| AndroidControl_3 | `AC_3` | `GUI-Model-AC_3` | ✓ (S1) | ✗ | id/ood × {state, action} 4 파일 | state_pred + action_pred ratio-mix 학습 (3:7, 5:5, 7:3) |
| MonkeyCollection | `MC` | `GUI-Model-MC` | ✓ (S1) | ✗ | 단일 test (random split) | 메타 없음 → Stage 2 자동 skip |
| MobiBench | `MB` | `GUI-Model-MB` | ✗ | ✗ | 단일 파일 (split 없음) | 평가 전용 |

### AndroidControl (AC)

대규모 모바일 제어 데이터셋. 본 실험의 주력. Stage 1/2 양쪽 모두에서 **app-level ID/OOD split** 을 단일 partition 으로 공유 — Stage 2 OOD 앱이 Stage 1 train 에 포함되지 않도록 보장한다.

| 항목 | 값 |
|------|-----|
| Stage 1 (World Modeling) | 71,047 건 |
| Stage 2 (Action Prediction) | 91,677 건 |
| Split 도구 | `scripts/split_data.py --dataset AndroidControl` (`primary_app` 기반 app-level) |

**Action Type 분포 (Stage 2)** — click 55.7% · finish 16.4% · scroll 11.9% · input 6.6% · open_app 6.1% · navigate_back 3.3% · long_click 0.2%.

### AndroidControl_2 (AC_2)

AC 와 schema 동일한 별도 분할 데이터. **단일 test (ID/OOD 분리 없음)** 로 사전 분할되어 제공되므로 `split_data.py` 를 다시 돌릴 필요가 없다. JSONL `images` 경로가 `AndroidControl/images/...` 를 참조하므로 AC 의 `images/` 디렉토리가 살아 있어야 한다 (별도 `images/` 없음).

| 항목 | 값 |
|------|-----|
| Stage 1 train / test | ~67K / ~3.5K (단일) |
| Stage 2 train / test | ~28K / ~1.5K (단일) |
| token budget | family default 2048 → AC_2 override **`max_tokens=5400`** (Cell 5 `_DATASET_CONFIG[..]["image_overrides"]`) |

평가는 `_hungarian_eval.py` / `_action_eval.py` 의 single-pair `overall` 모드 (1 섹션) 로 채점.

### AndroidControl_3 (AC_3)

AC 와 동일한 image 자산 위에서 **state_pred** (다음 화면 XML 예측) 와 **action_pred** (다음 액션 예측) 두 task 를 비율 혼합한 Stage 1 학습 데이터를 제공한다. 비율은 `state:action = 3:7, 5:5, 7:3` 3 종으로 분할되어 각각 다른 학습 가중치를 산출한다 (axis 추가). Test 는 task × split 조합 4 파일.

| 항목 | 값 |
|------|-----|
| Stage 1 train (ratio 3 종) | 각 ~50K (`gui-model_stage1_train_{3_7,5_5,7_3}.jsonl`) |
| Stage 1 test | id/ood × {state_pred, action_pred} (4 파일) |
| Split 도구 | `scripts/split_data.py --dataset AC_3 [--ac3-ratios 3_7,5_5,7_3 --ac3-train-total 50000]` |
| 채점 | state → `_hungarian_eval.py` (Stage1), action → `_action_eval.py` (Stage2) |
| HF slug | ratio 별 분기: `ac-3-r37-`, `ac-3-r55-`, `ac-3-r73-` |

> 학습 sweep: `--dataset AC_3` 가 ratio 3 종 모두를 자동으로 돌린다 (`--ac3-ratios r55,r73` 로 부분 실행 가능).
> 평가는 `--train-dataset AC_3 --ac3-ratio r55 --eval-datasets AC_3` 처럼 단일 ratio 의 가중치를 골라야 한다.

### MonkeyCollection (MC)

Stage 1 (world modeling) 전용 대규모 데이터셋 (약 100K). `episodes_meta.jsonl` 부재 → `split_data.py --dataset MC` 가 random split (기본 0.95) 만 수행. Stage 2 데이터가 존재하지 않으므로 노트북·스크립트 모두 `_STAGE1_ONLY` guard 로 Stage 2 파이프라인에서 자동 skip.

### MobiBench (MB) — 평가 전용 벤치마크

OOD 일반화 측정을 위한 단일 파일 벤치마크. `data/MobiBench/` 아래 `gui-model_stage1.jsonl` · `gui-model_stage2.jsonl` 두 단일 파일만 존재 (split 없음). `_action_eval.py` / `_hungarian_eval.py` 가 single-pair `overall` 1 섹션으로 채점. `dataset_info.json` 등록은 `scripts/_common.sh::ensure_eval_only_dataset_info()` 가 idempotent 하게 보장 → 노트북 미실행 환경에서도 평가 가능.

교차 평가 실행 (예):
```bash
bash scripts/stage1_eval.sh --model qwen3-vl-8b --train-dataset AC --eval-datasets AC,AC_2,MC,MB
bash scripts/stage2_eval.sh --model qwen3-vl-8b --train-dataset AC --eval-datasets AC,AC_2,MB \
     --stage1-mode full --stage1-epoch 3 --stage2-mode lora
```

> **학습 레시피**: 모든 학습 DS 가 3 epochs (baseline). Stage 1 / Stage 2 의 `full` · `lora` 학습 YAML 은 Section 0 의 YAML 일괄 생성 셀들 (Cell 8 = Stage 1, Cell 10 = Stage 2) 에서 자동 생성된다.
> **AC 하이퍼파라미터는 모델 크기 3 단(2B / 3-4B / 7-9B) 으로 공유**되며, `_SIZE_CONFIG_AC` (Cell 5) 에서 관리한다. AC_2 / MC 는 tier 미적용 — dataset baseline + per-model override 구조.


## 0. Environment Setup

프로젝트 경로 설정, GPU 확인, 의존성 설치를 수행합니다.

> **`BASE_DIR`을 본인의 프로젝트 경로로 수정하세요.**

In [ ]:
%%bash
set -e
# 0. 서브프로젝트 clone (없을 때만). uv 의 [tool.uv.sources] 가 ./LlamaFactory 를
#    editable path source 로 잡으므로 이 디렉토리가 존재해야 한다.
if [ ! -d LlamaFactory ]; then
  git clone https://github.com/hiyouga/LlamaFactory.git
fi

# 1. uv 로 .venv 생성 + 의존성 해소.
#    LlamaFactory editable + extras (transformers>=4.57.1,<4.58, vllm>=0.11.2,
#    deepspeed, av, fire, datasets, peft, trl, ...) 가 한 번에 설치된다.
#    LF transitive 상한 (transformers<=5.2.0) 과 우리 extras pin (>=4.57.1,<4.58)
#    가 4.57.x 구간에서 겹치므로 추가 우회 필요 없다.
uv venv --python 3.12
uv sync --extra llamafactory


### AC 크기-tier 하이퍼파라미터 (`_SIZE_CONFIG_AC`)

AC(AndroidControl) 은 모델 크기 3 단 (**2B / 3-4B / 7-9B**) 의 공유 하이퍼파라미터로 운영합니다. 아래 tier 값 위에 모델별 delta 만 `hparam_overrides` 로 얹힙니다. AC_2 (AndroidControl_2) · MC (MonkeyCollection) 는 tier 미적용 — dataset baseline + per-model override 구조 그대로. MB (MobiBench) 는 평가 전용 벤치마크이므로 학습 하이퍼파라미터 해석에서 제외됩니다.

**크기 배정 (총 8 모델)**:

| 구간 | 모델 |
|---|---|
| **2B** | qwen2-vl-2b |
| **3-4B** | qwen2.5-vl-3b · qwen3-vl-4b · qwen3.5-4b-base |
| **7-9B** | qwen2-vl-7b · qwen2.5-vl-7b · qwen3-vl-8b · qwen3.5-9b-base |

**Stage 1 (full FT)**:

| 구간 | lr | warmup_ratio | max_grad_norm |
|---|---|---|---|
| 2B | 1.5e-5 | 0.08 | 0.5 |
| 3-4B | 1.2e-5 | 0.06 | 0.5 |
| 7-9B | 1.0e-5 (baseline) | 0.03 (baseline) | 1.0 (baseline) |

**Stage 1 LoRA** (`stage1_full` 위에 덮어쓰기):

| 구간 | lr | LoRA r / α | dropout |
|---|---|---|---|
| 2B | 1.5e-4 | 8 / 16 | 0.05 |
| 3-4B | 1.2e-4 | 12 / 24 | 0.05 |
| 7-9B | 1.0e-4 | 16 / 32 | 0.05 |

**Stage 2 (LoRA)**:

| 구간 | lr | LoRA r / α | dropout | warmup | epochs |
|---|---|---|---|---|---|
| 2B | 6.0e-5 | 16 / 32 | 0.05 | 0.05 | 3 |
| 3-4B | 5.0e-5 | 24 / 48 | 0.05 | 0.04 | 3 |
| 7-9B | 4.0e-5 | 32 / 64 | 0.05 | 0.03 | 3 |

> **설계 근거**: `outputs/AC/eval/qwen{2.5-vl-7b,3-vl-8b}/stage2_eval` 실측에서 ① lr 5e-5 가 7-9B 에서 상단 경계 (7B e3 retrograde, 8B cond_text 퇴화), ② dropout 0.10 이 저빈도 action type (input 6.6%, nav_back 3.3%) 을 불안정하게 만듦이 확인됨. 2B / 3-4B 는 Stage 1 크기 규칙을 Stage 2 에 이식한 외삽.

> **참조 우선순위** (merge 순서): `_DATASET_CONFIG` baseline → `_SIZE_CONFIG_AC[size]` (AC 일 때만) → `_MODEL_CONFIG[model].hparam_overrides`. 아래 **Cell 5** 의 CONFIGS 빌더가 이 순서로 `dict.update()` 합니다.


In [ ]:
import os
from dotenv import load_dotenv

# ============================================================
# === Project root path ===
BASE_DIR = "./"
# ============================================================

BASE_DIR = os.path.abspath(BASE_DIR)
LF_ROOT = os.path.join(BASE_DIR, "LlamaFactory")

load_dotenv(os.path.join(BASE_DIR, ".env"))

# ============================================================
# === Global batch size & NPROC-aware grad_accum ===
# global batch 가 NPROC_PER_NODE 변경에 흔들리지 않도록
# gradient_accumulation_steps 를 자동 역계산한다.
#
#   global_batch = per_device_train_batch_size * grad_accum * NPROC_PER_NODE
#   grad_accum   = GLOBAL_BATCH_SIZE / (per_device * NPROC_PER_NODE)
# ============================================================
NPROC_PER_NODE = int(os.getenv("NPROC_PER_NODE", "2"))
GPU_TYPE = os.getenv("GPU_TYPE", "H100").upper()
GLOBAL_BATCH_SIZE = 64

_ALLOWED_GPU_TYPES = ("RTX5090", "A100", "H100")
_ALLOWED_NPROC = (1, 2, 4, 8)
if GPU_TYPE not in _ALLOWED_GPU_TYPES:
    raise ValueError(
        f"Unsupported GPU_TYPE={GPU_TYPE!r}. "
        f"Allowed: {_ALLOWED_GPU_TYPES}. Set in .env."
    )
if NPROC_PER_NODE not in _ALLOWED_NPROC:
    raise ValueError(
        f"Unsupported NPROC_PER_NODE={NPROC_PER_NODE}. "
        f"Allowed: {_ALLOWED_NPROC}. Set in .env."
    )

# === per_device_train_batch_size by (model size, GPU type) ============
# RTX5090=32GB, A100=80GB, H100=80GB. 8개 모델 모두 LlamaFactory backend.
# 값을 바꾸면 GLOBAL_BATCH_SIZE / (per_device * NPROC_PER_NODE) 가
# 정수가 되도록 유지해야 한다 (_derive_grad_accum 가 강제 검증).
_PER_DEVICE_BS_BY_SIZE = {
    "2B":   {"RTX5090": 4, "A100": 8, "H100": 8},
    "3-4B": {"RTX5090": 2, "A100": 4, "H100": 4},
    "7-9B": {"RTX5090": 1, "A100": 2, "H100": 2},
}


def lf_per_device_bs(size: str) -> int:
    """Return per_device_train_batch_size for given model size on current GPU."""
    return _PER_DEVICE_BS_BY_SIZE[size][GPU_TYPE]


def _derive_grad_accum(per_device: int, nproc: int = NPROC_PER_NODE,
                       target: int = GLOBAL_BATCH_SIZE) -> int:
    """Return gradient_accumulation_steps so that per_device * ga * nproc == target."""
    denom = per_device * nproc
    if denom <= 0 or target % denom != 0:
        raise ValueError(
            f"GLOBAL_BATCH_SIZE({target}) is not divisible by "
            f"per_device({per_device}) * NPROC_PER_NODE({nproc}) = {denom}. "
            f"Adjust NPROC_PER_NODE / GPU_TYPE in .env or GLOBAL_BATCH_SIZE."
        )
    return target // denom

# ============================================================
# === Model-family image-pixel configs ===
# Qwen 계열별 vision encoder patch-size (factor) 와 token budget.
# 각 family 의 max/min pixels 는 학습 YAML 의 image_max_pixels /
# image_min_pixels 필드에 그대로 주입된다. factor / merged_tokens /
# vertical_retention 은 데이터 전처리 메타이며 YAML 에는 들어가지 않는다.
#
# Token budget 은 학습 데이터셋에 따라 결정된다 (학습-추론 mismatch 방지).
#   - family default                 : max_tokens=2048   (AC, MC 학습 + 평가)
#   - AC2 (AndroidControl_2) override: max_tokens=5400   (AC2 학습 + 해당 모델의
#     모든 평가에 동일 budget 사용 — _common.sh::build_infer_cmd 가 TRAIN_DATASET
#     로 분기). family 별 factor 에 따라 max_pixels = max_tokens × factor² 로 환산.
#   Qwen2/2.5-VL  (factor 28): 2048→1,605,632, 5400→4,233,600
#   Qwen3-VL/3.5  (factor 32): 2048→2,097,152, 5400→5,529,600
# ============================================================
QWEN2_VL_CONFIG = {
    "max_pixels": 1_605_632,      # 2048 x 28²
    "min_pixels": 3_136,          # 4 x 28²
    "factor": 28,
    "max_tokens": 2048,
    "merged_tokens_at_1080x2400": 510,
    "vertical_retention": 0.782,
}
QWEN2_5_VL_CONFIG = {  # Qwen2-VL 과 동일
    "max_pixels": 1_605_632, "min_pixels": 3_136, "factor": 28, "max_tokens": 2048,
    "merged_tokens_at_1080x2400": 510, "vertical_retention": 0.782,
}
QWEN3_VL_CONFIG = {
    "max_pixels": 2_097_152,      # 2048 x 32²
    "min_pixels": 4_096,          # 4 x 32²
    "factor": 32,
    "max_tokens": 2048,
    "merged_tokens_at_1080x2400": 510,
    "vertical_retention": 0.893,
}
QWEN3_5_VL_CONFIG = {  # Qwen3-VL 과 동일 (사용자 지시)
    "max_pixels": 2_097_152, "min_pixels": 4_096, "factor": 32, "max_tokens": 2048,
    "merged_tokens_at_1080x2400": 510, "vertical_retention": 0.893,
}

MODEL_FAMILY_CONFIG = {
    "qwen2-vl-2b":     QWEN2_VL_CONFIG,
    "qwen2-vl-7b":     QWEN2_VL_CONFIG,
    "qwen2.5-vl-3b":   QWEN2_5_VL_CONFIG,
    "qwen2.5-vl-7b":   QWEN2_5_VL_CONFIG,
    "qwen3-vl-4b":     QWEN3_VL_CONFIG,
    "qwen3-vl-8b":     QWEN3_VL_CONFIG,
    "qwen3.5-4b-base": QWEN3_5_VL_CONFIG,
    "qwen3.5-9b-base": QWEN3_5_VL_CONFIG,
}

# ============================================================
# === Model Registry (8 models) ===
# AC / AC_2 전용으로 모델 크기 3 단 tier (2B / 3-4B / 7-9B) 의 공유 하이퍼파라미터
# (_SIZE_CONFIG_AC) 를 적용한다. MonkeyCollection / AndroidControl_3 는 tier 미적용
# (AC_3 는 dataset baseline + per-model override 만 적용; AC_3 ratio variant 3 키도 동일).
#
# 적용 우선순위 (merge 순서):
#   1. _DATASET_CONFIG[ds].stage{1,2}              (dataset baseline)
#   2. _SIZE_CONFIG_AC[size].stage{1, 1_lora, 2}   (AC 일 때만)
#   3. _MODEL_CONFIG[model].hparam_overrides       (계열 delta)
#
# image_max_pixels / image_min_pixels 는 MODEL_FAMILY_CONFIG 에서 자동 주입.
# cutoff_len: stage1=9216 (AC_3 RoPE 길이 회피), stage2=10000.
# ============================================================
def _img_cfg(short):
    f = MODEL_FAMILY_CONFIG[short]
    return {"image_max_pixels": f["max_pixels"], "image_min_pixels": f["min_pixels"]}

_MODEL_CONFIG = {
    "qwen2-vl-2b":   {"model_id": "Qwen/Qwen2-VL-2B-Instruct",   "short_name": "qwen2-vl-2b",   "template": "qwen2_vl",         "size": "2B",
        **_img_cfg("qwen2-vl-2b"),   "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen2-vl-7b":   {"model_id": "Qwen/Qwen2-VL-7B-Instruct",   "short_name": "qwen2-vl-7b",   "template": "qwen2_vl",         "size": "7-9B",
        **_img_cfg("qwen2-vl-7b"),   "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen2.5-vl-3b": {"model_id": "Qwen/Qwen2.5-VL-3B-Instruct", "short_name": "qwen2.5-vl-3b", "template": "qwen2_vl",         "size": "3-4B",
        **_img_cfg("qwen2.5-vl-3b"), "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen2.5-vl-7b": {"model_id": "Qwen/Qwen2.5-VL-7B-Instruct", "short_name": "qwen2.5-vl-7b", "template": "qwen2_vl",         "size": "7-9B",
        **_img_cfg("qwen2.5-vl-7b"), "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen3-vl-4b":   {"model_id": "Qwen/Qwen3-VL-4B-Instruct",   "short_name": "qwen3-vl-4b",   "template": "qwen3_vl_nothink", "size": "3-4B",
        **_img_cfg("qwen3-vl-4b"),   "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen3-vl-8b":   {"model_id": "Qwen/Qwen3-VL-8B-Instruct",   "short_name": "qwen3-vl-8b",   "template": "qwen3_vl_nothink", "size": "7-9B",
        **_img_cfg("qwen3-vl-8b"),   "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen3.5-4b-base": {"model_id": "Qwen/Qwen3.5-4B-Base", "short_name": "qwen3.5-4b-base", "template": "qwen3_5_nothink", "size": "3-4B",
        **_img_cfg("qwen3.5-4b-base"), "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen3.5-9b-base": {"model_id": "Qwen/Qwen3.5-9B-Base", "short_name": "qwen3.5-9b-base", "template": "qwen3_5_nothink", "size": "7-9B",
        **_img_cfg("qwen3.5-9b-base"), "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
}

# ============================================================
# === Dataset configs (baseline hparams per dataset) ===
# 학습 대상 DS 는 {AC, MC} 만. MB 는 평가 전용.
# ============================================================
# AC_3: state_pred / action_pred dual-task test. id/ood × task = 4 test 파일.
_DUAL_TASK_TEST = {"AndroidControl_3"}

# AC_3 ratio (state:action) → split_data.py 가 산출하는 train 파일 stem.
_AC3_RATIO_FILES = {
    "r37": "train_3_7",  # state 30% : action 70%
    "r55": "train_5_5",
    "r73": "train_7_3",
}

_DATASET_CONFIG = {
    "AndroidControl": {
        "lf_subfolder": "GUI-Model-AC",
        "ds_prefix": "GUI-Model-AC",
        "output_prefix": "AC/",
        "eval_model_suffix": "_ac",
        "hf_slug": "ac-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        "stage2": {
            "lr": "5.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 32, "lora_alpha": 64, "lora_dropout": 0.1,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
    },
    "AndroidControl_2": {
        "lf_subfolder": "GUI-Model-AC_2",
        "ds_prefix": "GUI-Model-AC_2",
        "output_prefix": "AC_2/",
        "eval_model_suffix": "_ac_2",
        "hf_slug": "ac-2-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        "stage2": {
            "lr": "5.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 32, "lora_alpha": 64, "lora_dropout": 0.1,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        # AC2 학습 시 image budget 을 5400 tokens 로 상향. family 별 factor 에
        # 따라 max_pixels 가 환산된다 (Qwen2/2.5-VL=4,233,600, Qwen3-VL/3.5=5,529,600).
        # min_tokens 는 family default(=4) 유지. 평가측 _common.sh::build_infer_cmd
        # 가 TRAIN_DATASET=AC_2 일 때 동일 budget 으로 추론한다.
        "image_overrides": {"max_tokens": 5400},
    },
    # AC_3 (AndroidControl_3) — state_pred + action_pred ratio-mix.
    # ratio 별로 별도 학습 가중치를 산출하므로 "AndroidControl_3_r{37,55,73}" 3 개의
    # 가상 DS 로 펼친다 (lf_subfolder/ds_prefix 는 'GUI-Model-AC_3' 공유, output/hf_slug 는 ratio 별).
    # NOTE: AC_3 은 Stage 1 전용 (Stage 2 학습 데이터 미정의) — _STAGE1_ONLY 가 보장.
    **{
        f"AndroidControl_3_{_r}": {
            "lf_subfolder": f"GUI-Model-AC_3_{_r}",
            "ds_prefix":    "GUI-Model-AC_3",          # dataset_info 의 entry 접두 (test 는 ratio 무관 공유)
            "output_prefix": "AC_3/",                  # ratio 별 산출물을 AC_3/ 단일 부모 아래로 모은다.
            "model_dir_suffix":   f"_{_r}",            # adapters/ / merged/ 모델 디렉토리에 ratio suffix.
            "eval_model_suffix":  f"_{_r}",            # eval/ 모델 디렉토리에도 동일 ratio suffix.
            "hf_slug":      f"ac-3-{_r}-",
            "ac3_ratio":    _r,
            "ac3_train_stem": _AC3_RATIO_FILES[_r],    # gui-model_stage1_{stem}.jsonl
            "stage1": {
                "lr": "1.0e-5", "epochs": 3, "warmup_ratio": 0.03,
                "save_strategy": "epoch", "save_steps": None,
                "eval_strategy": "epoch", "eval_steps": None,
                "per_device_eval_batch_size": 4,
                "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
                "weight_decay": 0.01, "max_grad_norm": 1.0,
                "lr_scheduler_type": "cosine",
            },
            # Stage 2 미정의 — _STAGE1_ONLY guard 로 skip.
            "stage2": {
                "lr": "5.0e-5", "epochs": 3, "warmup_ratio": 0.03,
                "save_strategy": "epoch", "save_steps": None,
                "eval_strategy": "epoch", "eval_steps": None,
                "per_device_eval_batch_size": 4,
                "lora_rank": 32, "lora_alpha": 64, "lora_dropout": 0.1,
                "weight_decay": 0.01, "max_grad_norm": 1.0,
                "lr_scheduler_type": "cosine",
            },
        }
        for _r in _AC3_RATIO_FILES
    },
    "MonkeyCollection": {
        "lf_subfolder": "GUI-Model-MC",
        "ds_prefix": "GUI-Model-MC",
        "output_prefix": "MC/",
        "hf_slug": "mc-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        # Stage 2 MC 미지원 — placeholder. `_STAGE1_ONLY` guard 로 skip.
        "stage2": {
            "lr": "5.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 32, "lora_alpha": 64, "lora_dropout": 0.1,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
    },
}

_STAGE1_ONLY = {"MonkeyCollection",
                "AndroidControl_3_r37", "AndroidControl_3_r55", "AndroidControl_3_r73"}

# ID/OOD split 없이 `gui-model_stage{1,2}_test.jsonl` 단일 파일을 사용하는 데이터셋.
# `_STAGE1_ONLY` 와 직교 — MC 는 Stage 1 만 + 단일 test, AC2 는 Stage 1+2 모두 단일 test.
_SINGLE_TEST = {"MonkeyCollection", "AndroidControl_2"}

_EVAL_ONLY_BENCHMARKS = {
    "MobiBench": {
        "ds_prefix": "GUI-Model-MB",
        "data_dir": os.path.join(BASE_DIR, "data", "MobiBench"),
        "stage1_jsonl": "gui-model_stage1.jsonl",
        "stage2_jsonl": "gui-model_stage2.jsonl",
        "ds_s1_name": "GUI-Model-MB_stage1",
        "ds_s2_name": "GUI-Model-MB_stage2",
    },
}

# ============================================================
# === Size-tier shared hparams (AC only) ===
# Stage 1 (full FT):
#   2B   : lr 1.5e-5, warmup 0.08, max_grad_norm 0.5
#   3-4B : lr 1.2e-5, warmup 0.06, max_grad_norm 0.5
#   7-9B : dataset baseline 유지
# Stage 1 LoRA (stage1_full 위에 추가 덮어쓰기):
#   2B   : lr 1.5e-4, r=8/a=16
#   3-4B : lr 1.2e-4, r=12/a=24
#   7-9B : lr 1.0e-4, r=16/a=32
# Stage 2 (LoRA):
#   2B   : lr 6.0e-5, warmup 0.05, r=16/a=32
#   3-4B : lr 5.0e-5, warmup 0.04, r=24/a=48
#   7-9B : lr 4.0e-5, r=32/a=64
# ============================================================
_SIZE_CONFIG_AC = {
    "2B": {
        "stage1":      {"lr": "1.5e-5", "warmup_ratio": 0.08, "max_grad_norm": 0.5},
        "stage1_lora": {"lr": "1.5e-4", "lora_rank": 8,  "lora_alpha": 16, "lora_dropout": 0.05},
        "stage2":      {"lr": "6.0e-5", "warmup_ratio": 0.05,
                        "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05},
    },
    "3-4B": {
        "stage1":      {"lr": "1.2e-5", "warmup_ratio": 0.06, "max_grad_norm": 0.5},
        "stage1_lora": {"lr": "1.2e-4", "lora_rank": 12, "lora_alpha": 24, "lora_dropout": 0.05},
        "stage2":      {"lr": "5.0e-5", "warmup_ratio": 0.04,
                        "lora_rank": 24, "lora_alpha": 48, "lora_dropout": 0.05},
    },
    "7-9B": {
        "stage1":      {},
        "stage1_lora": {"lr": "1.0e-4", "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05},
        "stage2":      {"lr": "4.0e-5", "lora_dropout": 0.05},
    },
}

# ============================================================
# === Model ordering for execution cells ===
# ============================================================
MODEL_ORDER = [
    ("qwen2-vl-2b",     "Qwen2-VL-2B"),
    ("qwen2-vl-7b",     "Qwen2-VL-7B"),
    ("qwen2.5-vl-3b",   "Qwen2.5-VL-3B"),
    ("qwen2.5-vl-7b",   "Qwen2.5-VL-7B"),
    ("qwen3-vl-4b",     "Qwen3-VL-4B"),
    ("qwen3-vl-8b",     "Qwen3-VL-8B"),
    ("qwen3.5-4b-base", "Qwen3.5-4B-Base"),
    ("qwen3.5-9b-base", "Qwen3.5-9B-Base"),
]
DS_ORDER = [("AC", "AndroidControl"), ("AC_2", "AndroidControl_2"), ("MC", "MonkeyCollection")]

# ============================================================
# === Build CONFIGS: CONFIGS[model_key][ds_name] ===
# ============================================================
CONFIGS = {}
for _model_key, _mcfg in _MODEL_CONFIG.items():
    CONFIGS[_model_key] = {}
    _overrides = _mcfg.get("hparam_overrides", {})
    _size = _mcfg["size"]
    _per_device = lf_per_device_bs(_size)
    _grad_accum = _derive_grad_accum(_per_device)
    for _ds_name, _cfg in _DATASET_CONFIG.items():
        c = dict(_cfg)
        # image budget: family default 위에 dataset image_overrides 를 덮어쓴다.
        # override 키는 token 단위 ("max_tokens", "min_tokens") 또는 px 단위
        # ("image_max_pixels", "image_min_pixels"). token 키는 family factor² 로 환산.
        _factor = MODEL_FAMILY_CONFIG[_model_key]["factor"]
        _img = dict(_img_cfg(_model_key))
        for _ok, _ov in _cfg.get("image_overrides", {}).items():
            if _ok == "max_tokens":
                _img["image_max_pixels"] = _ov * _factor * _factor
            elif _ok == "min_tokens":
                _img["image_min_pixels"] = _ov * _factor * _factor
            else:
                _img[_ok] = _ov
        c["image_max_pixels"] = _img["image_max_pixels"]
        c["image_min_pixels"] = _img["image_min_pixels"]
        c["model_key"] = _model_key
        c["model_id"] = _mcfg["model_id"]
        c["short_name"] = _mcfg["short_name"]
        c["template"] = _mcfg["template"]
        c["model_config"] = _mcfg
        c["dataset_name"] = _ds_name
        c["data_dir"] = os.path.join(BASE_DIR, "data", _ds_name)

        # AC_3 ratio variant 들은 동일한 data/AndroidControl_3 디렉토리를 공유.

        if _ds_name.startswith("AndroidControl_3"):

            c["data_dir"] = os.path.join(BASE_DIR, "data", "AndroidControl_3")
        # AC_3 ratio variant: train 은 ratio 별 분기, test 는 task × split 4 파일.
        if _ds_name in _DUAL_TASK_TEST or _ds_name.startswith("AndroidControl_3"):
            _r = _cfg.get("ac3_ratio", "")
            c["ds_s1_train"]            = f"{c['ds_prefix']}_stage1_train_{_r}" if _r else f"{c['ds_prefix']}_stage1_train"
            c["ds_s1_test_id_state"]    = f"{c['ds_prefix']}_stage1_test_id_state"
            c["ds_s1_test_ood_state"]   = f"{c['ds_prefix']}_stage1_test_ood_state"
            c["ds_s1_test_id_action"]   = f"{c['ds_prefix']}_stage1_test_id_action"
            c["ds_s1_test_ood_action"]  = f"{c['ds_prefix']}_stage1_test_ood_action"
            # Backwards-compat aliases (AC 흐름과 일치): test_id 는 state 를 default 로 노출.
            c["ds_s1_test_id"]          = c["ds_s1_test_id_state"]
            c["ds_s1_test_ood"]         = c["ds_s1_test_ood_state"]
            c["ds_s1_test"]             = c["ds_s1_test_id"]
        elif _ds_name in _SINGLE_TEST:
            c["ds_s1_train"] = f"{c['ds_prefix']}_stage1_train"
            c["ds_s1_test"]  = f"{c['ds_prefix']}_stage1_test"
        else:
            c["ds_s1_train"]    = f"{c['ds_prefix']}_stage1_train"
            c["ds_s1_test_id"]  = f"{c['ds_prefix']}_stage1_test_id"
            c["ds_s1_test_ood"] = f"{c['ds_prefix']}_stage1_test_ood"
            c["ds_s1_test"]     = c["ds_s1_test_id"]
        c["ds_s2_train"] = f"{c['ds_prefix']}_stage2_train"
        if _ds_name in _SINGLE_TEST:
            c["ds_s2_test"] = f"{c['ds_prefix']}_stage2_test"
        else:
            c["ds_s2_test_id"]  = f"{c['ds_prefix']}_stage2_test_id"
            c["ds_s2_test_ood"] = f"{c['ds_prefix']}_stage2_test_ood"
            c["ds_s2_test"]     = c["ds_s2_test_id"]

        c["hf_s1_model_full"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage1-full-world-model"
        c["hf_s1_model_lora"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage1-lora-world-model"
        c["hf_s1_model"] = c["hf_s1_model_full"]

        c["hf_s2_base"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage2-base"
        c["hf_s2_world_full"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage2-full-world-model"
        c["hf_s2_world_lora"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage2-lora-world-model"
        c["hf_s2_world"] = c["hf_s2_world_full"]

        _ds_code = c["output_prefix"].rstrip("/")
        _mshort  = _mcfg["short_name"]
        # AC_3 ratio variant: model dir 에 ratio suffix 를 붙여 outputs/AC_3/ 단일 부모 아래에서 충돌 방지.
        _mshort_dir = _mshort + c.get("model_dir_suffix", "")

        c["save_s1_full"]        = f"../outputs/{_ds_code}/adapters/{_mshort_dir}_stage1_full_world-model"
        c["save_s1_lora"]        = f"../outputs/{_ds_code}/adapters/{_mshort_dir}_stage1_lora_world-model"
        c["out_s1_merged_full"]  = f"../outputs/{_ds_code}/merged/{_mshort_dir}_stage1_full_world-model"
        c["out_s1_merged_lora"]  = f"../outputs/{_ds_code}/merged/{_mshort_dir}_stage1_lora_world-model"
        c["save_s1"]        = c["save_s1_full"]
        c["out_s1_merged"]  = c["out_s1_merged_full"]

        for _m2 in ("full", "lora"):
            c[f"save_s2_{_m2}_base"]                 = f"../outputs/{_ds_code}/adapters/{_mshort}_stage2_{_m2}_base"
            c[f"save_s2_{_m2}_world_from_full"]      = f"../outputs/{_ds_code}/adapters/{_mshort}_stage2_{_m2}_world-model_from_full-ep__STAGE1_EPOCH__"
            c[f"save_s2_{_m2}_world_from_lora"]      = f"../outputs/{_ds_code}/adapters/{_mshort}_stage2_{_m2}_world-model_from_lora-ep__STAGE1_EPOCH__"
            c[f"out_s2_merged_{_m2}_base"]           = f"../outputs/{_ds_code}/merged/{_mshort}_stage2_{_m2}_base"
            c[f"out_s2_merged_{_m2}_world_from_full"]= f"../outputs/{_ds_code}/merged/{_mshort}_stage2_{_m2}_world-model_from_full-ep__STAGE1_EPOCH__"
            c[f"out_s2_merged_{_m2}_world_from_lora"]= f"../outputs/{_ds_code}/merged/{_mshort}_stage2_{_m2}_world-model_from_lora-ep__STAGE1_EPOCH__"
        c["save_s2_base"]                 = c["save_s2_lora_base"]
        c["save_s2_world_from_full"]      = c["save_s2_lora_world_from_full"]
        c["save_s2_world_from_lora"]      = c["save_s2_lora_world_from_lora"]
        c["out_s2_merged_base"]           = c["out_s2_merged_lora_base"]
        c["out_s2_merged_world_from_full"]= c["out_s2_merged_lora_world_from_full"]
        c["out_s2_merged_world_from_lora"]= c["out_s2_merged_lora_world_from_lora"]
        c["save_s2_world"]                = c["save_s2_world_from_full"]
        c["out_s2_merged_world"]          = c["out_s2_merged_world_from_full"]

        _tier = _SIZE_CONFIG_AC[_size] if _ds_name in {"AndroidControl", "AndroidControl_2"} else None

        s1_full = dict(c["stage1"])
        if _tier is not None:
            s1_full.update(_tier.get("stage1", {}))
        s1_full.update(_overrides.get("stage1", {}))

        s1_lora = dict(s1_full)
        if _tier is not None:
            s1_lora.update(_tier.get("stage1_lora", {}))
        s1_lora.update(_overrides.get("stage1_lora", {}))

        s2 = dict(c["stage2"])
        if _tier is not None:
            s2.update(_tier.get("stage2", {}))
        s2.update(_overrides.get("stage2", {}))

        s1_full["gradient_accumulation_steps"] = _grad_accum
        s1_lora["gradient_accumulation_steps"] = _grad_accum
        s2["gradient_accumulation_steps"] = _grad_accum

        c["stage1_full"] = s1_full
        c["stage1_lora"] = s1_lora
        c["stage1"]      = s1_full
        c["stage2"]      = s2
        c["stage1_only"] = _ds_name in _STAGE1_ONLY
        c["per_device_train_batch_size"] = _per_device

        CONFIGS[_model_key][_ds_name] = c

print(f"Working directory: {os.getcwd()}")
print(f"LlamaFactory root: {LF_ROOT}")
print(f"GPU_TYPE={GPU_TYPE}, NPROC_PER_NODE={NPROC_PER_NODE}, GLOBAL_BATCH_SIZE={GLOBAL_BATCH_SIZE}")
print("  per_device_train_batch_size × grad_accum  (target global={}):".format(GLOBAL_BATCH_SIZE))
for _sz in ("2B", "3-4B", "7-9B"):
    _pd = lf_per_device_bs(_sz)
    _ga = _derive_grad_accum(_pd)
    print(f"    {_sz:5s}: {_pd} × {_ga} × {NPROC_PER_NODE} = {_pd * _ga * NPROC_PER_NODE}")
print(f"Models: {list(CONFIGS.keys())}")
print(f"Train datasets: {list(_DATASET_CONFIG.keys())}")
print(f"Stage1-only datasets: {sorted(_STAGE1_ONLY)}")
print(f"Eval-only benchmarks: {list(_EVAL_ONLY_BENCHMARKS.keys())}")

# --- AC size-tier summary (per model, AC only) ---
print("\n=== AndroidControl size-tier resolution ===")
_size_order = {"2B": 0, "3-4B": 1, "7-9B": 2}
for _mk in sorted(CONFIGS.keys(), key=lambda k: (_size_order[_MODEL_CONFIG[k]["size"]], k)):
    c = CONFIGS[_mk]["AndroidControl"]
    s1f, s1l, s2 = c["stage1_full"], c["stage1_lora"], c["stage2"]
    fam = MODEL_FAMILY_CONFIG[_mk]
    print(f"  {_mk:18s} size={_MODEL_CONFIG[_mk]['size']:5s}  "
          f"img={fam['max_pixels']}/f{fam['factor']}  "
          f"s1 lr={s1f['lr']} wu={s1f['warmup_ratio']}  "
          f"s2 lr={s2['lr']} r={s2['lora_rank']}/a={s2['lora_alpha']}/d={s2['lora_dropout']}")


### YAML Configs 일괄 생성

이 블록은 `LlamaFactory/examples/custom/` 하위 학습 YAML 을 **환경설정 직후 한 번에 생성**합니다. 이후 학습 단계에서 별도의 YAML 생성 셀 실행 없이 즉시 스크립트/CLI 를 호출할 수 있습니다.


**포함 대상 (12개 모델 × 2개 데이터셋)**

| 블록 | 생성 파일 | 비고 |
|------|-----------|------|
| `[LlamaFactory]` Stage 1 Training YAML | `custom/GUI-Model-{MB,AC}/stage1_{full,lora}/{MODEL}_world-model.yaml` | full / lora 양쪽 생성 (Cell 9) |
| `[LlamaFactory]` Stage 2 Training YAML | `custom/GUI-Model-{MB,AC}/stage2_{full,lora}/{MODEL}_{base,world-model-full,world-model-lora}.yaml` | full / lora × 3 variant = 6 YAML (Cell 13) |


**HF repo id 규칙** (`_common.sh`):
- Stage 1 : `SaFD-00/{short}-{slug}world-model-stage1-{full|lora}-epoch{E}`
- Stage 2 base : `SaFD-00/{short}-{slug}base-stage2-{full|lora}-epoch{E2}`
- Stage 2 world-model : `SaFD-00/{short}-{slug}world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}`


#### [LlamaFactory] Stage 1 Training YAML

`LlamaFactory/examples/custom/GUI-Model-{MB,AC}/stage1_{full,lora}/{MODEL}_world-model.yaml` 생성. LF_ROOT 기준 상대경로 (`../outputs/`) 사용. Qwen 계열 8개 모델 전체에 대해 생성된다.

> **DeepSpeed config (GPU 별 분기)**
> - `GPU_TYPE=RTX5090` (32GB VRAM, 1장): `examples/deepspeed/ds_z3_offload_config.json` 로 강제 swap — ZeRO-3 + CPU offload optimizer 로 단일 GPU OOM 회피 (mode 무관). LoRA 도 8B 모델의 frozen weight all-gather + vision tokens (≤2048) + cutoff_len 9216 activations 합으로 32GB 를 넘기 때문에 offload 가 필요하다.
> - 그 외 (`A100` / `H100`, 80GB VRAM): 모델별 `stage1_deepspeed` (기본 `examples/deepspeed/ds_z3_config.json`) 그대로 사용.
> - **주의**: `ds_z3_offload_config.json` 은 DeepSpeed 가 `CPUAdamBuilder` 를 JIT 컴파일하므로 **시스템 CUDA toolkit (nvcc + `cuda.h` 헤더 + `lib64`) 이 torch 의 cu 버전과 정확히 일치**해야 한다 — 불일치 시 `CUDAMismatchException` 으로 학습 시작 전 죽는다. `scripts/_common.sh` 가 RTX5090 한정으로 `CUDA_HOME=/usr/local/cuda` 정렬을 강제 (nvcc 버전 vs `torch.version.cuda` 검증) 하므로 cu12.8 toolkit 이 `/usr/local/cuda` 에 link 되어 있어야 한다.

In [ ]:
from pathlib import Path

# LF Stage 1 YAML — full / lora 두 벌 생성.
# LoRA mode 는 finetuning_type: lora + lora_rank/alpha/target/dropout 블록을 추가.
# DeepSpeed config: GPU_TYPE=RTX5090 이면 ds_z3_offload_config.json 으로 강제 swap (32GB 1장 OOM 회피, mode 무관).
#                   LoRA 도 frozen 8B weight all-gather + vision tokens + cutoff_len 9216 activations 합으로
#                   offload 없이는 OOM. CPUAdamBuilder JIT 의존성은 _common.sh 의 RTX5090 한정
#                   CUDA_HOME 정렬 가드 (nvcc vs torch.version.cuda) 로 보정한다.
#                   그 외 (A100/H100, 80GB) 는 mode 무관 mcfg.stage1_deepspeed 그대로 (기본 ds_z3_config.json).
# AC_3 는 ratio variant 3 키 (AndroidControl_3_r{37,55,73}) 를 갖고 lf_subfolder 가 ratio 별 (GUI-Model-AC_3_r{37,55,73}) 이라
# (8 모델 × 6 DS × 2 mode = 96 YAML) 자동 생성. dataset 필드는 GUI-Model-AC_3_stage1_train_{r37,r55,r73} 로 ratio 분기.
for MODE in ["full", "lora"]:
    for model_key, ds_configs in CONFIGS.items():
        for ds_name, cfg in ds_configs.items():
            s1 = cfg[f"stage1_{MODE}"]
            mcfg = cfg["model_config"]
            yaml_dir = Path(LF_ROOT) / "examples" / "custom" / cfg["lf_subfolder"] / f"stage1_{MODE}"
            yaml_dir.mkdir(parents=True, exist_ok=True)

            save_steps_line = f"save_steps: {s1['save_steps']}\n" if s1["save_strategy"] == "steps" else ""
            _ds_path = mcfg.get("stage1_deepspeed")
            if GPU_TYPE == "RTX5090":
                _ds_path = "examples/deepspeed/ds_z3_offload_config.json"
            ds_line = f"deepspeed: {_ds_path}\n" if _ds_path else ""
            optim_line = f"optim: {s1['optim']}\n" if s1.get("optim") else ""
            seed_line = f"seed: {s1['seed']}\n" if s1.get("seed") is not None else ""

            output_dir = cfg[f"save_s1_{MODE}"]

            if MODE == "full":
                method_block = f"""\
### method
stage: sft
do_train: true
finetuning_type: full
freeze_vision_tower: {str(mcfg["freeze_vision_tower"]).lower()}"""
            else:
                method_block = f"""\
### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: {str(mcfg["freeze_vision_tower"]).lower()}
lora_rank: {s1["lora_rank"]}
lora_alpha: {s1["lora_alpha"]}
lora_target: all
lora_dropout: {s1["lora_dropout"]}"""

            STAGE1_CONFIG = f"""\
### model
model_name_or_path: {cfg["model_id"]}
trust_remote_code: true
image_max_pixels: {cfg["image_max_pixels"]}
image_min_pixels: {cfg["image_min_pixels"]}

{method_block}

### dataset
dataset: {cfg["ds_s1_train"]}
template: {cfg["template"]}
cutoff_len: 9216
overwrite_cache: false
preprocessing_num_workers: 16
media_dir: ../data

### output
output_dir: {output_dir}
logging_steps: 1
save_strategy: {s1["save_strategy"]}
{save_steps_line}save_total_limit: 5
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: {cfg["per_device_train_batch_size"]}
gradient_accumulation_steps: {s1["gradient_accumulation_steps"]}
learning_rate: {s1["lr"]}
num_train_epochs: {s1["epochs"]}
lr_scheduler_type: {s1["lr_scheduler_type"]}
warmup_ratio: {s1["warmup_ratio"]}
weight_decay: {s1["weight_decay"]}
max_grad_norm: {s1["max_grad_norm"]}
{optim_line}{seed_line}bf16: true
gradient_checkpointing: true
{ds_line}ddp_timeout: 18000000
# resume_from_checkpoint: true
"""

            fpath = yaml_dir / f"{model_key}_world-model.yaml"
            with open(fpath, 'w') as f:
                f.write(STAGE1_CONFIG)
            print(f"[LF/{MODE}] {fpath.relative_to(Path(LF_ROOT))}")
            print(f"  model: {cfg['model_id']}, template: {cfg['template']}")
            print(f"  lr={s1['lr']}, warmup={s1['warmup_ratio']}, wd={s1['weight_decay']}, grad_norm={s1['max_grad_norm']}")
            if MODE == "lora":
                print(f"  lora: r={s1['lora_rank']}, alpha={s1['lora_alpha']}, dropout={s1['lora_dropout']}")
            print(f"  output_dir: {output_dir}")
            print()


#### [LlamaFactory] Stage 2 Training YAML

`LlamaFactory/examples/custom/GUI-Model-{MB,AC}/stage2_{full,lora}/{MODEL}_{base,world-model-full,world-model-lora}.yaml` 생성. world-model variant 는 Stage 1 merged 모델 (HF Hub) 을 base 로 지정하지만, `stage2_train.sh` 가 runtime 에 `model_name_or_path` 를 local `merged/{M}_stage1_${STAGE1_MODE}/epoch-${STAGE1_EPOCH}/` 경로로 sed 치환한다. LF_ROOT 기준 상대경로 (`../outputs/`) 사용. Qwen 계열 8개 모델 전체에 대해 생성된다.

- Full FT: `finetuning_type: full`, `learning_rate` 는 Full 안정화를 위해 자동으로 낮춤 (기본 1.5e-5, `CONFIGS[...]['stage2']['full_lr']` 로 조정 가능).
- LoRA: `finetuning_type: lora` + `lora_rank/alpha/target/dropout` 블록.

> **DeepSpeed config (Stage 1 과 동일 정책)**
> - `GPU_TYPE=RTX5090`: `examples/deepspeed/ds_z3_offload_config.json` 강제 swap (mode 무관).
> - 그 외 (`A100` / `H100`): 모델별 `stage1_deepspeed` (기본 `ds_z3_config.json`) 그대로 사용.
> - cu toolkit 정렬 요구사항 (`_common.sh` 의 RTX5090 가드) 은 Stage 1 markdown 참조.

In [ ]:
import os
from pathlib import Path

# Stage 2: full/lora 두 모드 x base / world-model-{full,lora} = 6 YAML / (model, dataset).
# DeepSpeed config: GPU_TYPE=RTX5090 이면 ds_z3_offload_config.json 으로 강제 swap (Stage 1 과 동일 정책, mode 무관).
#                   CUDA_HOME 정렬 가드는 _common.sh 의 RTX5090 분기 참조.
for MODE2 in ["full", "lora"]:
    for model_key, ds_configs in CONFIGS.items():
        for ds_name, cfg in ds_configs.items():
            # Stage 2 를 지원하지 않는 데이터셋 (MC) 은 skip.
            if ds_name in _STAGE1_ONLY:
                continue
            s2 = cfg["stage2"]
            mcfg = cfg["model_config"]
            yaml_dir = Path(LF_ROOT) / "examples" / "custom" / cfg["lf_subfolder"] / f"stage2_{MODE2}"
            yaml_dir.mkdir(parents=True, exist_ok=True)

            save_steps_line = f"save_steps: {s2['save_steps']}\n" if s2["save_strategy"] == "steps" else ""
            optim_line = f"optim: {s2['optim']}\n" if s2.get("optim") else ""
            seed_line = f"seed: {s2['seed']}\n" if s2.get("seed") is not None else ""
            _ds_path = mcfg.get("stage1_deepspeed")
            if GPU_TYPE == "RTX5090":
                _ds_path = "examples/deepspeed/ds_z3_offload_config.json"
            ds_line = f"deepspeed: {_ds_path}\n" if _ds_path else ""

            if MODE2 == "full":
                method_block = (
                    "### method\n"
                    "stage: sft\n"
                    "do_train: true\n"
                    "finetuning_type: full\n"
                    f"freeze_vision_tower: {str(mcfg['freeze_vision_tower']).lower()}"
                )
                # Full FT 는 LoRA 대비 lr 을 낮춰 안정화.
                lr_value = s2.get("full_lr", 1.5e-5)
            else:
                method_block = (
                    "### method\n"
                    "stage: sft\n"
                    "do_train: true\n"
                    "finetuning_type: lora\n"
                    f"freeze_vision_tower: {str(mcfg['freeze_vision_tower']).lower()}\n"
                    f"lora_rank: {s2['lora_rank']}\n"
                    f"lora_alpha: {s2['lora_alpha']}\n"
                    "lora_target: all\n"
                    f"lora_dropout: {s2['lora_dropout']}"
                )
                lr_value = s2["lr"]

            COMMON_CONFIG = f"""\
### model
model_name_or_path: {{model_name_or_path}}
trust_remote_code: true
image_max_pixels: {cfg["image_max_pixels"]}
image_min_pixels: {cfg["image_min_pixels"]}

{method_block}

### dataset
dataset: {cfg["ds_s2_train"]}
template: {cfg["template"]}
cutoff_len: 10000
overwrite_cache: false
preprocessing_num_workers: 16
media_dir: ../data

### output
output_dir: {{output_dir}}
logging_steps: 1
save_strategy: {s2["save_strategy"]}
{save_steps_line}save_total_limit: 5
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: {cfg["per_device_train_batch_size"]}
gradient_accumulation_steps: {s2["gradient_accumulation_steps"]}
learning_rate: {lr_value}
num_train_epochs: {s2["epochs"]}
lr_scheduler_type: {s2["lr_scheduler_type"]}
warmup_ratio: {s2["warmup_ratio"]}
weight_decay: {s2["weight_decay"]}
max_grad_norm: {s2["max_grad_norm"]}
{optim_line}{seed_line}bf16: true
gradient_checkpointing: true
{ds_line}ddp_timeout: 18000000
# resume_from_checkpoint: true
"""

            VARIANTS = {
                "base": {
                    "model_name_or_path": cfg["model_id"],
                    "output_dir": cfg[f"save_s2_{MODE2}_base"],
                },
                "world-model-full": {
                    "model_name_or_path": cfg["hf_s1_model_full"],
                    "output_dir": cfg[f"save_s2_{MODE2}_world_from_full"],
                },
                "world-model-lora": {
                    "model_name_or_path": cfg["hf_s1_model_lora"],
                    "output_dir": cfg[f"save_s2_{MODE2}_world_from_lora"],
                },
            }

            print(f"=== {model_key} / {ds_name} Stage 2 ({MODE2}) YAML ===")
            print(f"  lr={lr_value}, warmup={s2['warmup_ratio']}, wd={s2['weight_decay']}, grad_norm={s2['max_grad_norm']}")
            for variant, params in VARIANTS.items():
                content = COMMON_CONFIG.format(
                    model_name_or_path=params["model_name_or_path"],
                    output_dir=params["output_dir"],
                )
                fpath = yaml_dir / f"{model_key}_{variant}.yaml"
                with open(fpath, 'w') as f:
                    f.write(content)
                print(f"[LF/{MODE2}] {fpath.relative_to(Path(LF_ROOT))}")
                print(f"  base:   {params['model_name_or_path']}")
                print(f"  output: {params['output_dir']}")
                print()


## 1. Stage 1 Data Preparation

`data/` 디렉토리의 Stage 1 (World Modeling) 데이터를 Train/Test Split 후 LLaMA-Factory 에 등록합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)

| File | AndroidControl | MonkeyCollection | MobiBench (eval-only) |
|------|----------------|-------------------|-----------------------|
| gui-model_stage1.jsonl | 71,047건 | ~100,000건 | 3,145건 (벤치마크 단일 파일) |
| gui-model_stage1_train.jsonl | 50,000 (default) | ~95,000 (95%) | — |
| gui-model_stage1_test_id.jsonl | 3,000 (default) | — | — |
| gui-model_stage1_test_ood.jsonl | 3,000 (default) | — | — |
| gui-model_stage1_test.jsonl | — | ~5,000 (5%) | — |
| Images | (jsonl 내 참조) | (jsonl 내 참조) | (벤치마크 원본) |

**Split 전략:**
- **AndroidControl** — `episodes_meta.jsonl.primary_app` 기반 **app-level ID/OOD** split
  (Stage 2 와 동일 partition 공유 → Stage 2 OOD 앱이 Stage 1 train 에서도 제외).
  기본 크기: train=50,000 / test_id=3,000 / test_ood=3,000 (`--stage1-{train,test-id,test-ood}-size`).
- **MonkeyCollection** — 메타 없음 → random split (seed=42, `--stage1-ratio` 기본 0.95).
- 실행: `python scripts/split_data.py --dataset AndroidControl` (또는 `... MonkeyCollection`).

**MobiBench (eval-only):**
- `data/MobiBench/gui-model_stage1.jsonl` 단일 파일이 **평가 세트 전체**로 사용됩니다 (ID/OOD split 없음).
- `stage1_eval.sh --eval-datasets MB` 시 dataset_info 엔트리 `GUI-Model-MB_stage1` 로 로드.


In [ ]:
import json
from pathlib import Path

LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

dataset_info_path = LF_DATA_DIR / "dataset_info.json"
if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

# Dataset registration is model-independent; iterate _DATASET_CONFIG directly
# (학습 대상 DS 만 train/test 쌍으로 등록).
_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Data Registration ===\n")

    # AC   : train + test_id + test_ood (app-level ID/OOD split, Stage 2 partition 공유)
    # MC   : train + test (random split, 메타 없음)
    # AC2  : train + test (AC 와 동일 schema, 단일 test, ID/OOD 분리 없음)
    # AC_3 : train_{r37,r55,r73} (ratio mix) + test_{id,ood}_{state,action}_pred 4 파일 (dual-task).
    #        ratio variant 3 키가 같은 dataset_info entry 의 train_{rXX} 쪽만 다르게 등록한다.
    if ds_name in _DUAL_TASK_TEST or ds_name.startswith("AndroidControl_3"):
        # AC_3 ratio variant: ratio 별 train + (id, ood) × (state, action) test 4 파일.
        _stem = cfg["ac3_train_stem"]
        _r = cfg["ac3_ratio"]
        required = [
            (f"gui-model_stage1_{_stem}.jsonl", f'{cfg["ds_prefix"]}_stage1_train_{_r}'),
            ("gui-model_stage1_test_id_state_pred.jsonl",  cfg["ds_s1_test_id_state"]),
            ("gui-model_stage1_test_ood_state_pred.jsonl", cfg["ds_s1_test_ood_state"]),
            ("gui-model_stage1_test_id_action_pred.jsonl", cfg["ds_s1_test_id_action"]),
            ("gui-model_stage1_test_ood_action_pred.jsonl", cfg["ds_s1_test_ood_action"]),
        ]
    elif ds_name in _SINGLE_TEST:
        required = [
            ("gui-model_stage1_train.jsonl", cfg["ds_s1_train"]),
            ("gui-model_stage1_test.jsonl",  cfg["ds_s1_test"]),
        ]
    else:
        required = [
            ("gui-model_stage1_train.jsonl",   cfg["ds_s1_train"]),
            ("gui-model_stage1_test_id.jsonl", cfg["ds_s1_test_id"]),
            ("gui-model_stage1_test_ood.jsonl", cfg["ds_s1_test_ood"]),
        ]
    for fname, _ds_key in required:
        fpath = DATA_PATH / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"{fpath} not found. Run split first:\n"
                f"  python scripts/split_data.py --dataset {ds_name}"
            )
        with open(fpath, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[OK] {fname} ({count} entries, {fpath.stat().st_size / 1024 / 1024:.1f} MB)")

    # AC_3 ratio variant 의 실제 디렉토리는 'AndroidControl_3' 단일 (state/action 공유).
    _data_subdir = "AndroidControl_3" if ds_name.startswith("AndroidControl_3") else ds_name
    DATASETS_TO_REGISTER = {
        ds_key: {
            "file_name": f"../../data/{_data_subdir}/{fname}",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS,
        }
        for fname, ds_key in required
    }

    for name, config in DATASETS_TO_REGISTER.items():
        dataset_info[name] = config
        print(f"[Registered] {name} -> {config['file_name']}")

# --- Eval-only benchmarks (MobiBench) : 단일 파일 entry ---
# MB 는 학습에 사용하지 않지만 평가 (stage1_eval.sh --eval-datasets MB) 시
# LlamaFactory dataset_info 에 등록된 이름이 필요하다. ID/OOD split 없이
# gui-model_stage1.jsonl 전체를 단일 evaluation set 으로 사용.
# NOTE: scripts/_common.sh::ensure_eval_only_dataset_info() 가 동일 엔트리를
#       평가 시작 시 자동 보장한다. 이 셀은 노트북 워크플로에서 같은 결과를
#       미리 기록해두는 역할 (idempotent — 두 경로 모두 같은 JSON 을 쓴다).
for bench_name, bcfg in _EVAL_ONLY_BENCHMARKS.items():
    bpath = Path(bcfg["data_dir"]) / bcfg["stage1_jsonl"]
    print(f"\n{'='*60}")
    print(f"=== {bench_name} Stage 1 Benchmark Registration (eval-only) ===\n")
    if not bpath.exists():
        print(f"[skip] {bpath} not found — benchmark file not staged. "
              f"Stage 1 eval on {bench_name} will fail until this file exists.")
        continue
    with open(bpath, 'r') as f:
        count = sum(1 for _ in f)
    print(f"[OK] {bcfg['stage1_jsonl']} ({count} entries, {bpath.stat().st_size / 1024 / 1024:.1f} MB)")
    dataset_info[bcfg["ds_s1_name"]] = {
        "file_name": f"../../data/{bench_name}/{bcfg['stage1_jsonl']}",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS,
    }
    print(f"[Registered] {bcfg['ds_s1_name']} -> ../../data/{bench_name}/{bcfg['stage1_jsonl']}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)
print(f"\n[saved] {dataset_info_path}")


In [ ]:
import json
from pathlib import Path

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Dataset Statistics ===\n")

    # MC: _train/_test  AC: _train + _test_id/_test_ood  AC_3: _train_{ratio} + _test_{id,ood}_{state,action}_pred
    _filenames = [
        "gui-model_stage1.jsonl",
        "gui-model_stage1_train.jsonl",
        "gui-model_stage1_test.jsonl",
        "gui-model_stage1_test_id.jsonl",
        "gui-model_stage1_test_ood.jsonl",
    ]
    if ds_name.startswith("AndroidControl_3"):
        # ratio variant 3 개가 같은 data_dir 를 공유하므로 중복 출력 방지: r37 만 stats 표시.
        if not ds_name.endswith("_r37"):
            print("  (other ratio variants share data/AndroidControl_3 — see _r37 above)")
            continue
        _filenames = [
            "gui-model_stage1_state_pred.jsonl",
            "gui-model_stage1_action_pred.jsonl",
            "gui-model_stage1_train_3_7.jsonl",
            "gui-model_stage1_train_5_5.jsonl",
            "gui-model_stage1_train_7_3.jsonl",
            "gui-model_stage1_test_id_state_pred.jsonl",
            "gui-model_stage1_test_ood_state_pred.jsonl",
            "gui-model_stage1_test_id_action_pred.jsonl",
            "gui-model_stage1_test_ood_action_pred.jsonl",
        ]
    for fname in _filenames:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            sample = entries[0] if entries else None
            msg_count = len(sample['messages']) if sample else 0
            img_count = len(sample.get('images', [])) if sample else 0
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   Messages per sample: {msg_count}")
            print(f"   Images per sample: {img_count}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")
            print()

    img_dir = DATA_PATH / "images"
    if img_dir.exists():
        imgs = list(img_dir.glob("*.png"))
        total_size = sum(f.stat().st_size for f in imgs) / 1024 / 1024 / 1024
        print(f"  Image directory: {img_dir}")
        print(f"   Total images: {len(imgs)}")
        print(f"   Total size: {total_size:.2f} GB")

### AC_3 Split (state_pred + action_pred → ratio mix)

`scripts/split_data.py --dataset AC_3` 가 `data/AndroidControl_3/` 의
`gui-model_stage1_state_pred.jsonl` 와 `gui-model_stage1_action_pred.jsonl` 를
재료로 ratio 3 종 (`state:action = 3:7, 5:5, 7:3`) train + (id, ood) × (state, action) test 를 만든다.
기본 train 합계는 50,000 행 (`--ac3-train-total` 로 조정).


In [ ]:
# AC_3 split 산출물 생성. idempotent — 산출물이 이미 있으면 재실행 시 덮어쓴다.
# 기본 ratio: 3_7, 5_5, 7_3 (state:action). 부분 실행은 --ac3-ratios 로 골라서.
!python scripts/split_data.py --dataset AC_3 --ac3-ratios 3_7,5_5,7_3 --ac3-train-total 50000


## 2. Stage 2 Data Preparation

Stage 2 (Action Prediction) 데이터를 **app-level in-domain / out-of-domain split** 으로 분리한 뒤 LlamaFactory 에 등록합니다.

**Task**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: Stage 1 과 동일 (공유)

**학습 대상 DS:** AndroidControl(AC), AndroidControl_2(AC_2). MonkeyCollection(MC) 은 Stage 2 데이터/YAML 부재로 제외 — Stage 1 전용.

**Split 워크플로 (AC):**
1. 메타데이터 추출 — `scripts/extract_androidcontrol_metadata.py` 로 GCS TFRecord 의 `actions` 중 첫 `open_app` 의 `app_name` 을 추출해 `episodes_meta.jsonl` 생성.
2. Split — `scripts/split_data.py --dataset AndroidControl` 가 앱 집합을 **in-domain / out-of-domain** 으로 나눈 뒤 각 풀에서 action-type stratified 샘플링 (largest-remainder).

| 파일 | AndroidControl (기본) | AndroidControl_2 |
|------|----------------------|------------------|
| `gui-model_stage2_train.jsonl` | 50,000 | (사전 split 본 사용) |
| `gui-model_stage2_test_id.jsonl` / `test.jsonl` | 3,000 (id) | 단일 `_test.jsonl` |
| `gui-model_stage2_test_ood.jsonl` | 3,000 | — |

> AC_2 는 데이터셋 자체가 단일 `_test.jsonl` 구조를 갖는다 (`_SINGLE_TEST` set 으로 처리). `_common.sh::build_infer_cmd` 가 `TRAIN_DATASET=AC_2` 일 때 image budget 을 5400 tokens 로 자동 상향한다.

**MobiBench (eval-only, cross-benchmark):**
- `data/MobiBench/gui-model_stage2.jsonl` 단일 파일을 **single-pair overall** 로 채점합니다 (ID/OOD split 없음).
- dataset_info 엔트리: `GUI-Model-MB_stage2`. `stage2_eval.sh --eval-datasets MB` 시 `_action_eval.py score --test ... --pred ...` 경로로 `overall` 섹션 1개만 기록.

**Split 전략 (AC):**
- App 단위로 ID / OOD 분리 → 같은 앱 에피소드가 두 test set 에 섞이지 않는다.
- 각 분할 내부는 action type 비율을 원본과 맞추는 stratified subsampling.
- Random seed: 42.


In [ ]:
import json
from pathlib import Path

LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

dataset_info_path = LF_DATA_DIR / "dataset_info.json"
if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    # MC 는 Stage 2 데이터가 없으므로 skip.
    if ds_name in _STAGE1_ONLY:
        print(f"[skip] {ds_name}: Stage 1 전용 데이터셋 — Stage 2 entries 등록하지 않음.")
        continue
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Data Registration ===\n")

    # AC  : train + test_id + test_ood (app-level ID/OOD split)
    # AC2 : train + test (단일 test, ID/OOD 분리 없음)
    if ds_name in _SINGLE_TEST:
        required = [
            ("gui-model_stage2_train.jsonl", cfg["ds_s2_train"]),
            ("gui-model_stage2_test.jsonl",  cfg["ds_s2_test"]),
        ]
    else:
        required = [
            ("gui-model_stage2_train.jsonl",    cfg["ds_s2_train"]),
            ("gui-model_stage2_test_id.jsonl",  cfg["ds_s2_test_id"]),
            ("gui-model_stage2_test_ood.jsonl", cfg["ds_s2_test_ood"]),
        ]
    for fname, _ds_key in required:
        fpath = DATA_PATH / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"{fpath} not found. Run split first:\n"
                f"  python scripts/extract_androidcontrol_metadata.py\n"
                f"  python scripts/split_data.py --dataset {ds_name}"
            )
        with open(fpath, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[OK] {fname} ({count} entries, {fpath.stat().st_size / 1024 / 1024:.1f} MB)")

    DATASETS_TO_REGISTER = {
        ds_key: {
            "file_name": f"../../data/{ds_name}/{fname}",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS,
        }
        for fname, ds_key in required
    }

    for name, config in DATASETS_TO_REGISTER.items():
        dataset_info[name] = config
        print(f"[Registered] {name} -> {config['file_name']}")

# --- Eval-only benchmarks (MobiBench) : 단일 파일 entry (single-pair 평가) ---
# MB Stage 2 는 ID/OOD split 없이 gui-model_stage2.jsonl 전체를 overall 1-섹션
# single-pair 모드로 채점한다 (_action_eval.py score --test --pred).
# NOTE: scripts/_common.sh::ensure_eval_only_dataset_info() 가 동일 엔트리를
#       평가 시작 시 자동 보장한다. 이 셀은 노트북 워크플로에서 같은 결과를
#       미리 기록해두는 역할 (idempotent — 두 경로 모두 같은 JSON 을 쓴다).
for bench_name, bcfg in _EVAL_ONLY_BENCHMARKS.items():
    bpath = Path(bcfg["data_dir"]) / bcfg["stage2_jsonl"]
    print(f"\n{'='*60}")
    print(f"=== {bench_name} Stage 2 Benchmark Registration (eval-only) ===\n")
    if not bpath.exists():
        print(f"[skip] {bpath} not found — benchmark file not staged. "
              f"Stage 2 eval on {bench_name} will fail until this file exists.")
        continue
    with open(bpath, 'r') as f:
        count = sum(1 for _ in f)
    print(f"[OK] {bcfg['stage2_jsonl']} ({count} entries, {bpath.stat().st_size / 1024 / 1024:.1f} MB)")
    dataset_info[bcfg["ds_s2_name"]] = {
        "file_name": f"../../data/{bench_name}/{bcfg['stage2_jsonl']}",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS,
    }
    print(f"[Registered] {bcfg['ds_s2_name']} -> ../../data/{bench_name}/{bcfg['stage2_jsonl']}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)
print(f"\n[saved] {dataset_info_path}")


In [ ]:
import json
from pathlib import Path
from collections import Counter

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    if ds_name in _STAGE1_ONLY:
        continue
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Dataset Statistics ===\n")

    for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

            action_types = []
            for entry in entries:
                try:
                    action = json.loads(entry['messages'][-1]['value'])
                    action_types.append(action.get('type', 'unknown'))
                except:
                    action_types.append('parse_error')

            type_counts = Counter(action_types)
            total = len(action_types)
            print(f"   Action type distribution:")
            for atype, count in type_counts.most_common():
                print(f"     {atype}: {count} ({count/total:.1%})")
            print()

In [ ]:
import json
from pathlib import Path
from collections import Counter

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    if ds_name in _STAGE1_ONLY:
        continue
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Action Type Analysis ===")

    train_path = DATA_PATH / "gui-model_stage2_train.jsonl"
    test_path = DATA_PATH / "gui-model_stage2_test.jsonl"

    for label, fpath in [("Train", train_path), ("Test", test_path)]:
        if not fpath.exists():
            print(f"[SKIP] {label}: {fpath.name} not found")
            continue

        with open(fpath, 'r') as f:
            entries = [json.loads(line) for line in f if line.strip()]

        action_types = []
        for entry in entries:
            try:
                action = json.loads(entry['messages'][-1]['value'])
                action_types.append(action.get('type', 'unknown'))
            except:
                action_types.append('parse_error')

        type_counts = Counter(action_types)
        total = len(action_types)
        print(f"\n  {label} Set Action Types ({total} entries)")
        for atype, count in type_counts.most_common():
            print(f"    {atype}: {count} ({count/total:.1%})")

## 3. Stage 1 SFT Training (Full FT, `qwen3-vl-8b`)

Stage 1 (World Modeling) 학습은 `scripts/stage1_train.sh` 가 담당한다. 본 노트북은 **`qwen3-vl-8b` + `--stage1-mode full`** 단일 변형 walkthrough 만 cell 로 둔다 — 다른 모델 / LoRA 모드는 동일 명령에서 `--model` / `--stage1-mode` 인자만 바꾸면 된다.

**사전 조건**
- `.env` 의 `HF_TOKEN` (Section 4 merge push 용), `NPROC_PER_NODE` (default 2)
- Cell 8 (Stage 1 YAML 일괄 생성) 이 `LlamaFactory/examples/custom/GUI-Model-{DS}/stage1_full/` 에 학습 YAML 작성
- Cell 12 (AC), Cell 13 (AC_2), Cell 15 (AC_3), Cell 14 (MC) 가 `LlamaFactory/data/dataset_info.json` 에 데이터셋 등록

### `scripts/stage1_train.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--model` | `all` | `scripts/_common.sh::MODELS` 에 등록된 short name (`qwen3-vl-8b`, `qwen2.5-vl-3b`, …) 또는 `all` |
| `--dataset` | `all` | `AC` · `AC_2` · `AC_3` · `MC` · `all` (AC_3 은 `--ac3-ratios` 별 sweep) |
| `--stage1-mode` | `full` | `full` (전 파라미터) 또는 `lora` (PEFT) |
| `--ac3-ratios` | `r37,r55,r73` | AC_3 한정. ratio 별 별도 학습 가중치 산출 |

내부적으로 `FORCE_TORCHRUN=1 NPROC_PER_NODE=$NPROC_PER_NODE llamafactory-cli train ...` 를 model × dataset 곱집합으로 호출한다 (DeepSpeed Z3).

**산출물**: `outputs/{DS}/adapters/{MODEL}_stage1_{MODE}_world-model/checkpoint-{N}/` (epoch 별 checkpoint).

도움말: `bash scripts/stage1_train.sh --help`.

### 3.1 AC

In [ ]:
!bash scripts/stage1_train.sh --model qwen3-vl-8b --dataset AC --stage1-mode full

### 3.2 AC_2

In [ ]:
!bash scripts/stage1_train.sh --model qwen3-vl-8b --dataset AC_2 --stage1-mode full

### 3.3 AC_3 (state + action ratio mix)

AC_3 는 `state_pred` / `action_pred` 두 task 를 ratio (state:action) 로 혼합한 train 3 종을 만든다 — ratio 별 학습 가중치가 다르므로 한 번의 호출이 `r37 / r55 / r73` 세 학습을 sweep 한다. `--ac3-ratios r55` 식으로 부분 실행도 가능.

In [ ]:
!bash scripts/stage1_train.sh --model qwen3-vl-8b --dataset AC_3 --stage1-mode full --ac3-ratios r37,r55,r73

### 다른 변형 실행

노트북에는 cell 을 추가하지 말고 인자만 바꿔 실행한다.

- **다른 모델**: `--model qwen2.5-vl-3b` 등. 등록된 모델 리스트는 `scripts/_common.sh::MODELS`.
- **LoRA mode**: `--stage1-mode lora`. 산출물 경로는 `..._stage1_lora_world-model/`.
- **여러 모델 sweep**: `--model all` (8 모델 전부).

## 4. Stage 1 Merge & Upload to HuggingFace (전 epoch push)

`scripts/stage1_merge.sh` 가 Section 3 의 `outputs/{DS}/adapters/.../checkpoint-*/` 를 epoch 별로 export 해 merged weight 를 만들고 HuggingFace Hub 에 push 한다. 본 노트북은 Section 3 와 동일하게 **`qwen3-vl-8b` + `full`** 단일 변형 walkthrough 만 둔다.

### `scripts/stage1_merge.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--model` | `all` | Section 3 와 동일 |
| `--dataset` | `all` | Section 3 와 동일 |
| `--stage1-mode` | `full` | `full` 은 checkpoint 자체를 export, `lora` 는 base + adapter 결합 |
| `--ac3-ratios` | `r37,r55,r73` | AC_3 한정 |

각 `checkpoint-*/` 의 `trainer_state.json.epoch` 을 파싱해 epoch 번호를 추출, 임시 export YAML 을 생성한 뒤 `llamafactory-cli export` 로 merged 디렉토리를 만들고 `huggingface_hub` 로 push (`HF_TOKEN` 필요).

**산출물**
- 로컬: `outputs/{DS}/merged/{MODEL}_stage1_{MODE}_world-model/epoch-{E}/`
- HF Hub: `SaFD-00/{short}-{slug}world-model-stage1-{MODE}-epoch{E}` (예: `SaFD-00/qwen3-vl-8b-ac-world-model-stage1-full-epoch1`)

도움말: `bash scripts/stage1_merge.sh --help`.

### 4.1 AC

In [ ]:
!bash scripts/stage1_merge.sh --model qwen3-vl-8b --dataset AC --stage1-mode full

### 4.2 AC_2

In [ ]:
!bash scripts/stage1_merge.sh --model qwen3-vl-8b --dataset AC_2 --stage1-mode full

### 4.3 AC_3 (ratio mix)

AC_3 는 ratio 별 (`r37`, `r55`, `r73`) 로 별도의 HF repo (`...ac-3-r37-world-model-stage1-full-epoch{E}` 등) 가 만들어진다.

In [ ]:
!bash scripts/stage1_merge.sh --model qwen3-vl-8b --dataset AC_3 --stage1-mode full --ac3-ratios r37,r55,r73

### 다른 변형 실행

- **다른 모델**: `--model {short_name}` (Section 3 와 동일).
- **LoRA mode**: `--stage1-mode lora` — Stage 3 에서 LoRA 로 학습한 adapter 가 있어야 한다.
- **여러 모델 sweep**: `--model all`.

### 5. Stage 1 Evaluation Pipeline

**흐름: `train → merge → eval`.** eval 은 HF Hub 에 push 된 merged repo (`SaFD-00/{short}-{slug}world-model-stage1-{full|lora}-epoch{E}`) 를 pull 해 sweep 한다. 로컬 `adapters/` · `merged/` 디렉토리에 의존하지 않으므로 학습 머신이 아닌 환경에서도 재평가가 성립한다.

1. **Baseline (zero-shot)** — `--variants base` 로 `{model_id}` 자체를 EVAL_DS 마다 추론. 산출: `outputs/{TRAIN_DS}/eval/{MODEL}/stage1_eval/base/on-{EVAL_DS}/hungarian_metrics.json`.
2. **HF Hub world-model variant sweep** — `--epochs LIST` (기본 `1,2,3`) 로 epoch 별 merged repo 를 pull → `vllm_infer.py` 추론 → `_hungarian_eval.py score` 로 metric 산출.
3. **without_open_app 자동 산출** — 각 `(variant, EVAL_DS)` 마다 정규 score 직후 추론 재실행 없이 `_hungarian_eval.py score --exclude-action open_app --filtered-test-dir ... --filtered-pred-dir on-{EVAL_DS}-without-open_app/` 가 한 번 더 호출되어 GT `open_app` 행을 양쪽에서 동시 drop 한 메트릭과 필터된 jsonl + `predict_results.json` 을 sibling 디렉토리에 idempotent 저장.
4. **EVAL_DS 별 분기** — AC 는 ID + OOD 2 파일 → `overall` / `in_domain` / `out_of_domain` 3 섹션. AC_2 / MC / MB 는 단일 파일 → single-pair `overall` 1 섹션. **AC_3 는 dual-task** — state_pred / action_pred 두 task 가 각각 (id, ood) 2 파일을 갖고, `_hungarian_eval.py` (state) + `_action_eval.py` (action) 두 채점기가 별도로 호출되어 `on-AC_3-state/hungarian_metrics.json` + `on-AC_3-action/action_metrics.json` 두 산출물을 만든다. AC_3 학습 모델 평가는 `--ac3-ratio {r37|r55|r73}` 로 ratio 가 정확히 한 개여야 한다.

산출 경로: `outputs/{TRAIN_DS}/eval/{MODEL}/stage1_eval/{variant}[/epoch-{E}]/on-{EVAL_DS}/hungarian_metrics.json`. 재실행 시 marker 존재 unit 은 정규 / without_open_app 각각 독립 skip.

> **자동 winner 선정 없음** — 사용자가 결과를 보고 Stage 2 에 사용할 epoch 을 `--stage1-epoch` 로 직접 지정 (Section 6 / 7 / 8).


### `scripts/stage1_eval.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--train-dataset` | (필수) | 단일 학습 DS — `AC` · `AC_2` · `AC_3` · `MC`. AC_3 은 `--ac3-ratio` 필수 |
| `--eval-datasets` | train DS | 콤마 구분 — `AC,AC_2,AC_3,MC,MB` |
| `--model` | `all` | 평가 대상 모델 short name |
| `--stage1-mode` | `full` | HF repo slug 결정 (`...stage1-{MODE}-epoch{E}`) |
| `--variants` | `base,full_world_model,lora_world_model` | `base` (zero-shot), `full_world_model` / `lora_world_model` (HF Hub merged repo pull) |
| `--epochs` | `1,2,3` | merged repo epoch sweep |
| `--ac3-ratio` | `r55` | AC_3 단일 ratio (학습 가중치 1 개 ↔ 평가 1 회) |

AC_3 는 dual-task — `state_pred` 는 `_hungarian_eval.py` (Stage 1 채점), `action_pred` 는 `_action_eval.py` (Stage 2 채점) 로 독립 처리하며 산출 디렉토리도 `on-AC_3-state/` · `on-AC_3-action/` 로 분리된다.

**산출물**: `outputs/{TRAIN_DS}/eval/{MODEL}/stage1_eval/{variant}[/epoch-{E}]/on-{EVAL_DS}/{hungarian_metrics.json | action_metrics.json}` (AC 는 ID/OOD 두 jsonl, 그 외는 단일 파일).

도움말: `bash scripts/stage1_eval.sh --help`. 아래 cell 들은 본 노트북이 정의하는 variant matrix · plotting 코드로, shell 호출은 단일 모델 smoke test 와 cross-benchmark 예시만 둔다.

In [ ]:
import json, math, os
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt

# Stage 1 Eval-Loss report — baseline vs per-checkpoint world-model sweep.
# 최신 epoch 기준으로 trainer eval_loss 를 함께 표로 뽑아 참고용 차트 생성.

def _load_eval_loss(p: Path):
    if not p.exists():
        return None
    try:
        with open(p, "r") as f:
            return json.load(f).get("eval_loss")
    except Exception:
        return None

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        ds_code    = cfg["output_prefix"].rstrip("/")
        eval_loss_dir = Path(BASE_DIR) / "outputs" / ds_code / "eval" / (short_name + cfg.get("eval_model_suffix", "")) / "stage1_eval" / "eval_loss"
        base_loss = _load_eval_loss(eval_loss_dir / "base" / "eval_results.json")
        if base_loss is None:
            print(f"[SKIP] {model_key}/{ds_name}: baseline eval_results.json 없음")
            continue

        # 두 모드 각각 per-checkpoint 집계
        mode_rows = {}
        for MODE in ("full", "lora"):
            wm_dir = eval_loss_dir / f"{MODE}_world_model"
            per_ckpt = []
            if wm_dir.is_dir():
                for cp in sorted(wm_dir.glob("epoch-*"),
                                 key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1):
                    l = _load_eval_loss(cp / "eval_results.json")
                    if l is not None:
                        per_ckpt.append((cp.name, l))
            if not per_ckpt:
                continue

            # winner: BEST_CHECKPOINT 참고 (adapter 디렉토리)
            best_file = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}{cfg.get('model_dir_suffix', '')}_stage1_{MODE}" / "BEST_CHECKPOINT"
            winner_name = best_file.read_text().strip() if best_file.exists() else per_ckpt[-1][0]
            mode_rows[MODE] = {"checkpoints": per_ckpt, "winner": winner_name}

        if not mode_rows:
            print(f"[SKIP] {model_key}/{ds_name}: no world-model checkpoint eval_results 발견")
            continue

        base_ppl = math.exp(base_loss)
        print(f"\n=== {model_key} / {ds_name} Stage 1 Eval-Loss ===")
        print(f"Baseline (Zero-shot)  eval_loss={base_loss:.4f}  ppl={base_ppl:.4f}")
        for MODE, info in mode_rows.items():
            print(f"[{MODE}_world_model]")
            for name, l in info["checkpoints"]:
                mark = " <-- winner" if name == info["winner"] else ""
                print(f"  {name:<18} eval_loss={l:.4f}  ppl={math.exp(l):.4f}{mark}")

        # Winner checkpoint 를 대표로 써서 막대그래프 — baseline vs winner (모드별)
        variants = [("Base\n(Zero-shot)", base_loss)]
        for MODE, info in mode_rows.items():
            w_loss = dict(info["checkpoints"])[info["winner"]]
            variants.append((f"{MODE}\n({info['winner']})", w_loss))
        labels, losses = zip(*variants)
        colors = ["#9E9E9E"] + ["#FF5722", "#3F51B5"][:len(variants) - 1]
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        b0 = axes[0].bar(labels, losses, color=colors, width=0.5)
        axes[0].set_title("Eval Loss")
        axes[0].set_ylabel("Cross-Entropy Loss")
        for bar, val in zip(b0, losses):
            axes[0].text(bar.get_x()+bar.get_width()/2, val+max(losses)*0.02, f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
        axes[0].set_ylim(0, max(losses) * 1.3); axes[0].grid(axis="y", alpha=0.3)
        perps = [math.exp(l) for l in losses]
        b1 = axes[1].bar(labels, perps, color=colors, width=0.5)
        axes[1].set_title("Perplexity"); axes[1].set_ylabel("Perplexity = exp(loss)")
        for bar, val in zip(b1, perps):
            axes[1].text(bar.get_x()+bar.get_width()/2, val+max(perps)*0.02, f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
        axes[1].set_ylim(0, max(perps) * 1.3); axes[1].grid(axis="y", alpha=0.3)
        plt.suptitle(f"Stage 1 Eval-Loss ({model_key} / {ds_name}) — baseline vs winner", fontsize=13, fontweight="bold")
        plt.tight_layout()
        img_path = eval_loss_dir / "stage1_eval_loss_evaluation.png"
        plt.savefig(img_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"[Saved] {img_path}")

        # Markdown report
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        rlines = [f"# Stage 1 Eval-Loss Report: Base vs per-checkpoint sweep ({model_key} / {ds_name})",
                  f"\n> Generated: {now}\n",
                  "## Experiment Setup\n",
                  "| Item | Value |",
                  "|------|-------|",
                  f"| Model | {cfg['model_id']} |",
                  f"| Dataset | {ds_name} |",
                  f"| Test Dataset | {cfg['ds_s1_test']} |",
                  ""]
        rlines.append("## Baseline\n")
        rlines += ["| Metric | Base (Zero-shot) |",
                   "|--------|------------------|",
                   f"| Eval Loss | {base_loss:.4f} |",
                   f"| Perplexity | {base_ppl:.4f} |", ""]
        for MODE, info in mode_rows.items():
            rlines.append(f"## World-Model (stage1_{MODE}) — per-checkpoint\n")
            rlines += [f"| Checkpoint | Eval Loss | Perplexity | vs Base |",
                       f"|------------|-----------|------------|---------|"]
            for name, l in info["checkpoints"]:
                mark = "  **(winner)**" if name == info["winner"] else ""
                rlines.append(f"| {name}{mark} | {l:.4f} | {math.exp(l):.4f} | {(base_loss - l)/base_loss:+.1%} |")
            rlines.append("")
        report = "\n".join(rlines)
        report_path = eval_loss_dir / "evaluation_report.md"
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(report, encoding="utf-8")
        print(f"[Saved] {report_path}")


In [ ]:
## Stage 1 Baseline (zero-shot) — base model 을 각 평가 DS 에 대해 추론.
## 학습 DS 와 무관한 단일 baseline 이므로 --train-dataset 은 AC 로 고정 (산출 경로만 달라짐).
## 평가 DS: AC, MC (in-distribution), MB (cross-dataset 벤치마크), AC_3 (dual-task 채점).
## NOTE: AC_3 는 EVAL_DS 로 들어갈 때 ratio 와 무관 (test 4 파일은 ratio 공유).
##       단, --train-dataset AC_3 자체는 별도 셀 (AC_3 Eval 섹션) 에서 처리.
##
## 산출: outputs/AC/eval/{MODEL}/stage1_eval/base/on-{EVAL_DS}/hungarian_metrics.json
!bash scripts/stage1_eval.sh --train-dataset AC --eval-datasets AC,MC,MB,AC_3 --variants base
## (MC 학습 모델의 baseline 도 별도 기록하려면 주석 해제)
# !bash scripts/stage1_eval.sh --train-dataset MC --eval-datasets AC,MC,MB --variants base


In [ ]:
## Stage 1 world-model variant HF Hub sweep.
##
## AC / MC 두 학습 DS 각각에서 HF 에 push 된 merged repo 를
##   SaFD-00/{short}-{slug}world-model-stage1-{full|lora}-epoch{E}
## 를 pull 해 AC / MC (in-distribution) + MB / AC_3 (cross-benchmark) 에서 평가.
## AC_3 는 dual-task (state + action) 두 산출물이 따로 생성된다.
##
## 산출: outputs/{TRAIN_DS}/eval/{MODEL}/stage1_eval/{full|lora}_world_model/epoch-{E}/on-{EVAL_DS}/hungarian_metrics.json
##
## Winner 자동 선정은 없음. 결과를 보고 Stage 2 의 --stage1-epoch 를 수동 선택.
!bash scripts/stage1_eval.sh --train-dataset AC --eval-datasets AC,MC,MB,AC_3 \
    --variants full_world_model,lora_world_model --epochs 1,2,3

!bash scripts/stage1_eval.sh --train-dataset MC --eval-datasets AC,MC,MB,AC_3 \
    --variants full_world_model,lora_world_model --epochs 1,2,3


In [ ]:
# NOTE: 이 셀은 scripts/_hungarian_eval.py 와 글자 단위 동치를 유지하는 정본 복제다.
# 실제 평가는 `python scripts/_hungarian_eval.py score ...` 또는 stage1_eval.sh 를 통해 실행한다.
# 이 셀은 로직 레퍼런스 / 빠른 디버깅 용도로만 유지된다.

import json
import re
import math
from collections import Counter
from bs4 import BeautifulSoup
from munkres import Munkres

# ── Hungarian Metric 상수 ─────────────────────────────────────────────────
INTERACTIVE_TAGS = {"button", "input", "a", "select", "textarea"}
CONTENT_TAGS     = {"p", "img", "span"}
CLICKABLE_ATTRS  = {"clickable", "long-clickable"}

W_TAG   = 3.0   # tag 불일치 패널티 (매칭 불가)
W_TEXT  = 1.5   # text 불일치
W_INDEX = 0.2   # DOM index 거리

MATCH_THRESHOLD = 1.5   # 이 이상이면 매칭 거부
INDEX_TAU       = 2     # index_acc: 차이 ≤ τ 이면 위치 정확


# ── 요소 추출 (BeautifulSoup) ─────────────────────────────────────────────

def _collect_texts(el):
    """요소 자신 + 자식 전체에서 텍스트 토큰 수집 (중복 제거, 알파벳순 join)."""
    tokens = set()
    def add(v):
        if v:
            tokens.add(v.strip())
    add(el.get("description"))
    add(el.get("id"))
    for child in el.find_all(True):
        add(child.get("description"))
        add(child.get("id"))
        t = child.get_text(strip=True)
        if t:
            tokens.add(t)
    t = el.get_text(strip=True)
    if t:
        tokens.add(t)
    return " | ".join(sorted(tokens)) if tokens else ""


def _safe_int(v, default=-1):
    try:
        return int(v)
    except (TypeError, ValueError):
        return default


def extract_elements(xml_str):
    """XML/HTML 문자열에서 interactive 요소를 평탄화하여 추출."""
    try:
        soup = BeautifulSoup(xml_str, "xml")
    except Exception:
        soup = BeautifulSoup(xml_str, "html.parser")
    elements = []
    for el in soup.find_all(True):
        tag  = el.name
        idx  = _safe_int(el.get("index", -1))
        text = _collect_texts(el)
        is_interactive = tag in INTERACTIVE_TAGS
        is_content     = (tag in CONTENT_TAGS) and bool(text)
        is_clickable   = any(el.get(a) for a in CLICKABLE_ATTRS)
        if is_interactive or is_content or is_clickable:
            elements.append({"tag": tag, "text": text, "index": idx})
    return elements


# ── 비용 함수 ─────────────────────────────────────────────────────────────

def _text_sim(a, b):
    """Jaccard 유사도 (단어 집합 기준)."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    sa = set(a.lower().replace("|", "").split())
    sb = set(b.lower().replace("|", "").split())
    return len(sa & sb) / len(sa | sb)


def _match_cost(e1, e2, max_idx):
    """pred 요소 e1과 gt 요소 e2 사이의 매칭 비용."""
    if e1["tag"] != e2["tag"]:
        return W_TAG
    tc = W_TEXT  * (1.0 - _text_sim(e1["text"], e2["text"]))
    ic = W_INDEX * (abs(e1["index"] - e2["index"]) / max(max_idx, 1))
    return round(tc + ic, 5)


# ── 헝가리안 매칭 ─────────────────────────────────────────────────────────

def _hungarian_match(pred, gt):
    """헝가리안 알고리즘으로 pred-gt 간 최적 1:1 매칭."""
    n, m = len(pred), len(gt)
    if n == 0 or m == 0:
        return [], []
    max_idx = max(
        (e["index"] for e in pred + gt if e["index"] >= 0),
        default=1,
    )
    matrix = [
        [_match_cost(p, g, max_idx) for g in gt]
        for p in pred
    ]
    size   = max(n, m)
    padded = [row + [MATCH_THRESHOLD * 2] * (size - len(row)) for row in matrix]
    while len(padded) < size:
        padded.append([MATCH_THRESHOLD * 2] * size)
    indexes = Munkres().compute(padded)
    pairs = []
    for i, j in indexes:
        if i < n and j < m and matrix[i][j] < MATCH_THRESHOLD:
            pairs.append((i, j, matrix[i][j]))
    return pairs, matrix


def compute_hungarian_acc(pred_str, gt_str):
    """모델 생성 XML과 정답 XML을 비교해 hungarian 기반 평가 메트릭을 반환."""
    _zero = {
        "hungarian_ea": 0.0, "hungarian_f1": 0.0,
        "hungarian_prec": 0.0, "hungarian_rec": 0.0,
        "hungarian_text": 0.0, "hungarian_idx": 0.0,
    }
    try:
        pred_els = extract_elements(pred_str)
        gt_els   = extract_elements(gt_str)
    except Exception:
        return _zero
    if not gt_els:
        return _zero

    pairs, _ = _hungarian_match(pred_els, gt_els)
    n_pred, n_gt, n_matched = len(pred_els), len(gt_els), len(pairs)

    ea   = n_matched / max(n_pred, n_gt) if max(n_pred, n_gt) > 0 else 0.0
    prec = n_matched / n_pred             if n_pred  > 0           else 0.0
    rec  = n_matched / n_gt               if n_gt    > 0           else 0.0
    f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0    else 0.0

    if pairs:
        text_sims = [_text_sim(pred_els[i]["text"], gt_els[j]["text"]) for i, j, _ in pairs]
        idx_diffs = [abs(pred_els[i]["index"] - gt_els[j]["index"]) for i, j, _ in pairs]
        text_avg  = sum(text_sims) / len(text_sims)
        idx_acc   = sum(1 for d in idx_diffs if d <= INDEX_TAU) / len(idx_diffs)
    else:
        text_avg = 0.0
        idx_acc  = 0.0

    return {
        "hungarian_ea":   round(ea, 4),
        "hungarian_f1":   round(f1, 4),
        "hungarian_prec": round(prec, 4),
        "hungarian_rec":  round(rec, 4),
        "hungarian_text": round(text_avg, 4),
        "hungarian_idx":  round(idx_acc, 4),
    }


# ── 텍스트 생성 품질 메트릭 ───────────────────────────────────────────────

def calc_bleu(reference, hypothesis, max_n=4):
    """BLEU-4 score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not hyp_tokens or not ref_tokens:
        return 0.0

    bp = min(1.0, math.exp(1 - len(ref_tokens) / len(hyp_tokens)))

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1))
        hyp_ngrams = Counter(tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens) - n + 1))

        clipped = sum(min(count, ref_ngrams.get(ng, 0)) for ng, count in hyp_ngrams.items())
        total = sum(hyp_ngrams.values())

        if total == 0:
            precisions.append(0)
        else:
            precisions.append(clipped / total)

    if any(p == 0 for p in precisions):
        return 0.0

    log_avg = sum(math.log(p) for p in precisions) / max_n
    return bp * math.exp(log_avg)


def calc_rouge_l(reference, hypothesis):
    """ROUGE-L (F1) score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return 0.0

    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs_len = dp[m][n]
    precision = lcs_len / n
    recall = lcs_len / m

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ── Stage 1 전체 평가 함수 ────────────────────────────────────────────────

def evaluate_stage1_predictions(test_path, pred_path):
    """Stage 1 prediction 전체 평가를 수행합니다."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_text = gt_entry['messages'][-1]['value']
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))

        bleu = calc_bleu(gt_text, pred_text)
        rouge_l = calc_rouge_l(gt_text, pred_text)
        exact_match = 1.0 if gt_text.strip() == pred_text.strip() else 0.0
        hungarian = compute_hungarian_acc(pred_text, gt_text)

        results.append({
            'bleu': bleu, 'rouge_l': rouge_l,
            'exact_match': exact_match, 'hungarian': hungarian,
        })

    total = len(results)
    metrics = {
        'total': total,
        'avg_bleu': sum(r['bleu'] for r in results) / total if total else 0,
        'avg_rouge_l': sum(r['rouge_l'] for r in results) / total if total else 0,
        'exact_match_rate': sum(r['exact_match'] for r in results) / total if total else 0,
        'avg_hungarian_ea':   sum(r['hungarian']['hungarian_ea'] for r in results) / total if total else 0,
        'avg_hungarian_f1':   sum(r['hungarian']['hungarian_f1'] for r in results) / total if total else 0,
        'avg_hungarian_prec': sum(r['hungarian']['hungarian_prec'] for r in results) / total if total else 0,
        'avg_hungarian_rec':  sum(r['hungarian']['hungarian_rec'] for r in results) / total if total else 0,
        'avg_hungarian_text': sum(r['hungarian']['hungarian_text'] for r in results) / total if total else 0,
        'avg_hungarian_idx':  sum(r['hungarian']['hungarian_idx'] for r in results) / total if total else 0,
    }
    return metrics, results

print("[Loaded] Stage 1 evaluation functions: calc_bleu, calc_rouge_l, compute_hungarian_acc, evaluate_stage1_predictions")

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path

all_stage1_metrics = {}

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        op = cfg["output_prefix"]
        s1_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / (short_name + cfg.get("eval_model_suffix", "")) / "stage1_eval"
        hung_dir = s1_eval_dir / "hungarian_matching"
        # AC 는 ID/OOD split 이라 GT 가 두 파일. 이 셀은 in-notebook 빠른 보기용으로
        # in-domain 파일을 anchor 로 쓴다 (canonical eval 은 stage1_eval.sh + _hungarian_eval.py).
        if ds_name in _STAGE1_ONLY:
            test_path = Path(cfg["data_dir"]) / "gui-model_stage1_test.jsonl"
        else:
            test_path = Path(cfg["data_dir"]) / "gui-model_stage1_test_id.jsonl"
        train_dir = Path(BASE_DIR) / cfg["save_s1"].lstrip("../")

        MODEL_PREDS = {}
        base_pred = hung_dir / "base" / "generated_predictions.jsonl"
        if base_pred.exists():
            MODEL_PREDS["Baseline (Zero-shot)"] = base_pred
        for p in sorted(hung_dir.glob("epoch-*/generated_predictions.jsonl"),
                        key=lambda x: int(x.parent.name.split("-")[1])):
            MODEL_PREDS[p.parent.name] = p

        combo_key = f"{model_key}/{ds_name}"
        print(f"\n{'='*70}")
        print(f"=== {combo_key} Stage 1 Evaluation ({len(MODEL_PREDS)} model(s)) ===")

        base_loss_path = s1_eval_dir / "eval_loss" / "base" / "eval_results.json"
        fwm_loss_path  = s1_eval_dir / "eval_loss" / "full_world_model" / "eval_results.json"
        if base_loss_path.exists() and fwm_loss_path.exists():
            with open(base_loss_path, 'r') as f: base_loss_metrics = json.load(f)
            with open(fwm_loss_path,  'r') as f: fwm_loss_metrics  = json.load(f)
            print(f"\n  Loss-based Evaluation:")
            print(f"    Base     eval_loss: {base_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(base_loss_metrics['eval_loss']):.4f}")
            print(f"    FWM      eval_loss: {fwm_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(fwm_loss_metrics['eval_loss']):.4f}")

        ds_metrics = {}
        for model_name, pred_path in MODEL_PREDS.items():
            if not pred_path.exists():
                print(f"  [SKIP] {model_name}: {pred_path} missing")
                continue
            metrics, _ = evaluate_stage1_predictions(str(test_path), str(pred_path))
            ds_metrics[model_name] = metrics
            out_metric_path = pred_path.parent / "hungarian_metrics.json"
            with open(out_metric_path, 'w', encoding='utf-8') as f:
                json.dump(metrics, f, ensure_ascii=False, indent=2)

        all_stage1_metrics[combo_key] = ds_metrics

        if ds_metrics:
            print(f"\n  === {combo_key} Stage 1 Comparison ===")
            comparison = pd.DataFrame({
                name: {
                    'BLEU-4': f"{m['avg_bleu']:.4f}", 'ROUGE-L': f"{m['avg_rouge_l']:.4f}",
                    'Exact Match': f"{m['exact_match_rate']:.1%}", 'Hungarian EA': f"{m['avg_hungarian_ea']:.4f}",
                    'Hungarian F1': f"{m['avg_hungarian_f1']:.4f}", 'Hungarian Prec': f"{m['avg_hungarian_prec']:.4f}",
                    'Hungarian Rec': f"{m['avg_hungarian_rec']:.4f}", 'Hungarian Text': f"{m['avg_hungarian_text']:.4f}",
                    'Hungarian Idx': f"{m['avg_hungarian_idx']:.4f}",
                } for name, m in ds_metrics.items()
            })
            display(comparison)

            ckpt_metrics = {k: v for k, v in ds_metrics.items() if k.startswith("checkpoint-")}
            if ckpt_metrics:
                best_name, best_m = max(ckpt_metrics.items(),
                                        key=lambda kv: (kv[1]['avg_hungarian_f1'],
                                                        int(kv[0].split('-')[1])))
                train_dir.mkdir(parents=True, exist_ok=True)
                (train_dir / "BEST_CHECKPOINT").write_text(best_name + "\n", encoding='utf-8')
                summary = {
                    "selected": best_name,
                    "metric": "avg_hungarian_f1",
                    "metric_value": best_m['avg_hungarian_f1'],
                    "candidates": [
                        {"checkpoint": k, "avg_hungarian_f1": v['avg_hungarian_f1'],
                         "avg_bleu": v['avg_bleu'], "avg_rouge_l": v['avg_rouge_l'],
                         "exact_match_rate": v['exact_match_rate']}
                        for k, v in ckpt_metrics.items()
                    ],
                }
                (train_dir / "BEST_CHECKPOINT.json").write_text(
                    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding='utf-8')
                print(f"\n  [{combo_key}] Hungarian F1 winner: {best_name} (F1={best_m['avg_hungarian_f1']:.4f})")
                print(f"     -> {train_dir / 'BEST_CHECKPOINT'}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

for combo_key, ds_metrics in all_stage1_metrics.items():
    if not ds_metrics:
        continue
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    model_names = list(ds_metrics.keys())
    n_models = len(model_names)
    colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    text_metrics = ['avg_bleu', 'avg_rouge_l', 'exact_match_rate']
    text_labels = ['BLEU-4', 'ROUGE-L', 'Exact Match']
    x1 = np.arange(len(text_metrics))
    width = 0.8 / n_models

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in text_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[0].bar(x1 + offset, values, width, label=name, color=colors[i])

    axes[0].set_xlabel('Metric'); axes[0].set_ylabel('Score')
    axes[0].set_title('Text Generation Metrics')
    axes[0].set_xticks(x1); axes[0].set_xticklabels(text_labels, rotation=15)
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.0); axes[0].grid(axis='y', alpha=0.3)

    hung_metrics = ['avg_hungarian_ea', 'avg_hungarian_f1', 'avg_hungarian_prec',
                    'avg_hungarian_rec', 'avg_hungarian_text', 'avg_hungarian_idx']
    hung_labels = ['EA', 'F1', 'Precision', 'Recall', 'Text Sim', 'Index Acc']
    x2 = np.arange(len(hung_metrics))

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in hung_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])

    axes[1].set_xlabel('Metric'); axes[1].set_ylabel('Score')
    axes[1].set_title('Hungarian Element Matching')
    axes[1].set_xticks(x2); axes[1].set_xticklabels(hung_labels, rotation=15)
    axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1.0); axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(f'Stage 1 Evaluation ({combo_key})', fontsize=14, fontweight='bold')
    plt.tight_layout()

    op = cfg["output_prefix"]
    s1_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / (short_name + cfg.get("eval_model_suffix", "")) / "stage1_eval"
    save_path = str(s1_eval_dir / "hungarian_matching" / "stage1_hungarian_matching_evaluation.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {save_path}")

In [ ]:
import json, math
from pathlib import Path
from datetime import datetime

# Stage 1 Hungarian Matching report — baseline + per-checkpoint sweep.
# 사전 준비: all_stage1_metrics[f"{model_key}/{ds_name}"] 은 다음 구조여야 한다.
#   {
#     "base":                 <metrics dict>,
#     "full_world_model":     {"<ckpt>": <metrics dict>, ...},   # optional
#     "lora_world_model":     {"<ckpt>": <metrics dict>, ...},   # optional
#   }
# (Cell 57 이 per-checkpoint hungarian_metrics.json 을 이 구조로 적재하도록 갱신됨.)

def _winner_ckpt(adapter_dir: Path, fallback_key: str):
    bc = adapter_dir / "BEST_CHECKPOINT"
    if bc.exists():
        return bc.read_text().strip()
    return fallback_key

def _metric_row(label: str, m: dict, keys):
    cells = [f"{m[key]:{fmt}}" if key in m else "-" for key, _lbl, fmt in keys]
    return f"| {label} | " + " | ".join(cells) + " |"

METRIC_KEYS = [
    ("avg_bleu",          "BLEU-4",         ".4f"),
    ("avg_rouge_l",       "ROUGE-L",        ".4f"),
    ("exact_match_rate",  "Exact Match",    ".1%"),
    ("avg_hungarian_ea",  "Hungarian EA",   ".4f"),
    ("avg_hungarian_f1",  "Hungarian F1",   ".4f"),
    ("avg_hungarian_prec","Hungarian Prec", ".4f"),
    ("avg_hungarian_rec", "Hungarian Rec",  ".4f"),
    ("avg_hungarian_text","Hungarian Text", ".4f"),
    ("avg_hungarian_idx", "Hungarian Idx",  ".4f"),
]

for combo_key, entries in all_stage1_metrics.items():
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    ds_code    = cfg["output_prefix"].rstrip("/")
    s1_eval_dir = Path(BASE_DIR) / "outputs" / ds_code / "eval" / (short_name + cfg.get("eval_model_suffix", "")) / "stage1_eval"
    base_metrics = entries.get("base")

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rlines = [f"# Stage 1 Hungarian Matching Report ({model_key} / {ds_name})",
              f"\n> Generated: {now}\n",
              "## Experiment Setup\n",
              "| Item | Value |",
              "|------|-------|",
              f"| Model | {cfg['model_id']} |",
              f"| Dataset | {ds_name} |",
              f"| Training | {cfg['stage1']['epochs']} epochs, LR={cfg['stage1']['lr']} (cosine) |",
              ""]
    if base_metrics:
        rlines.append("## Baseline (Zero-shot)\n")
        rlines += ["| Metric | Score |", "|--------|-------|"]
        for key, lbl, fmt in METRIC_KEYS:
            if key in base_metrics:
                rlines.append(f"| {lbl} | {base_metrics[key]:{fmt}} |")
        rlines.append("")

    for MODE in ("full", "lora"):
        ckpt_map = entries.get(f"{MODE}_world_model")
        if not ckpt_map:
            continue
        adapter_dir = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}{cfg.get('model_dir_suffix', '')}_stage1_{MODE}"
        winner = _winner_ckpt(adapter_dir, max(ckpt_map.keys(),
                              key=lambda c: int(c.split("-")[-1]) if c.split("-")[-1].isdigit() else -1))
        rlines.append(f"## World-Model (stage1_{MODE}) — per-checkpoint sweep\n")
        header = ["Checkpoint"] + [lbl for _, lbl, _ in METRIC_KEYS]
        rlines.append("| " + " | ".join(header) + " |")
        rlines.append("|" + "|".join(["--------"] * len(header)) + "|")
        for name, m in sorted(ckpt_map.items(),
                              key=lambda kv: int(kv[0].split("-")[-1]) if kv[0].split("-")[-1].isdigit() else -1):
            mark = "  **(winner)**" if name == winner else ""
            row = [name + mark] + [f"{m[key]:{fmt}}" if key in m else "-" for key, _, fmt in METRIC_KEYS]
            rlines.append("| " + " | ".join(row) + " |")
        rlines.append("")

    report = "\n".join(rlines)
    report_path = s1_eval_dir / "hungarian_matching" / "evaluation_report.md"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(report, encoding="utf-8")
    print(f"[Saved] {report_path}")

print("\nDone.")


### Stage 1 AC_3 Eval (state + action 독립 채점)

AC_3 학습 모델의 평가는 ratio 가 정확히 한 개여야 한다 (`--ac3-ratio r55` default).
task 별 산출물:
* `outputs/AC_3_rXX/eval/{model}/stage1_eval/{variant}/epoch-N/on-AC_3-state/hungarian_metrics.json`
* `outputs/AC_3_rXX/eval/{model}/stage1_eval/{variant}/epoch-N/on-AC_3-action/action_metrics.json`


In [ ]:
# AC_3 ratio = r55 학습 결과를 AC_3 test 로 평가 (8 모델 × full+lora world-model + base).
!bash scripts/stage1_eval.sh --train-dataset AC_3 --ac3-ratio r55 --eval-datasets AC_3


In [ ]:
# 단일 모델 / 단일 ratio smoke test.
!bash scripts/stage1_eval.sh --model qwen2-vl-2b --train-dataset AC_3 --ac3-ratio r55 \
    --eval-datasets AC_3 --variants base --epochs 1


In [ ]:
# AC_3 학습 모델을 AC / MB 등 다른 벤치마크에도 교차 평가 (ratio 1 개 고정).
!bash scripts/stage1_eval.sh --train-dataset AC_3 --ac3-ratio r73 \
    --eval-datasets AC_3,AC,MB


## 6. Stage 2 SFT Training (LoRA, `qwen3-vl-8b`)

Stage 2 (Action Prediction) 학습은 `scripts/stage2_train.sh` 가 담당한다. Stage 2 는 **`base`** (Stage 1 우회) 와 **`world-model-{stage1_mode}`** (Section 4 의 Stage 1 merged epoch 를 상류로 사용) 두 variant 를 지원하며, 본 노트북은 **`qwen3-vl-8b` + `--stage2-mode lora` + 상류 Stage 1 `full` epoch 1**  단일 변형 walkthrough 만 둔다.

**사전 조건**
- Cell 10 (Stage 2 YAML 일괄 생성) 이 `LlamaFactory/examples/custom/GUI-Model-{DS}/stage2_lora/` 에 학습 YAML 작성
- world-model variant 을 쓰려면 Section 4 가 끝나 있어야 함 (`outputs/{DS}/merged/.../epoch-{E1}/` 존재)
- AC_3 / MC 는 Stage 2 미지원 (`AC` · `AC_2` 만 가능)

### `scripts/stage2_train.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--model` | `all` | Section 3 와 동일한 short name 레지스트리 |
| `--dataset` | `all` | `AC` · `AC_2` · `all` |
| `--stage2-mode` | `lora` | `lora` (PEFT) 또는 `full` |
| `--stage1-mode` | `full` | 상류 Stage 1 산출 repo 의 mode (`full` / `lora`) |
| `--stage1-epoch` | (world-model variant 시 필수) | 어떤 Stage 1 epoch 의 merged 를 상류로 쓸지 |

world-model variant 은 runtime 에 YAML 의 `model_name_or_path` 를 `outputs/{DS}/merged/.../epoch-{E1}/` 으로 sed 치환, `__STAGE1_EPOCH__` placeholder 를 epoch 번호로 치환한 뒤 학습.

**산출물**: `outputs/{DS}/adapters/{MODEL}_stage2_{variant_key}/checkpoint-{N}/` (`variant_key` = `world-model-{stage1_mode}-epoch-{E1}` 또는 `base`).

도움말: `bash scripts/stage2_train.sh --help`.

### 6.1 AC

In [ ]:
!bash scripts/stage2_train.sh --model qwen3-vl-8b --dataset AC --stage2-mode lora --stage1-mode full --stage1-epoch 1

### 6.2 AC_2

In [ ]:
!bash scripts/stage2_train.sh --model qwen3-vl-8b --dataset AC_2 --stage2-mode lora --stage1-mode full --stage1-epoch 1

### 다른 변형 실행

- **다른 모델**: `--model {short_name}`.
- **Full FT**: `--stage2-mode full`.
- **다른 Stage 1 계보**: `--stage1-mode lora --stage1-epoch 3` 식으로 상류 epoch / mode 변경.
- **base variant** (Stage 1 우회): `--stage1-epoch` 생략 → YAML 의 base 모델로 직접 학습.

## 7. Stage 2 Merge & Upload to HuggingFace (전 epoch push)

`scripts/stage2_merge.sh` 가 Section 6 의 checkpoint 들을 epoch 별 merged 로 export + HF Hub push 한다. Section 6 와 동일하게 **`qwen3-vl-8b` + `lora` + 상류 Stage 1 `full` epoch 1** 단일 변형 walkthrough.

### `scripts/stage2_merge.sh` 사용법

인자는 `stage2_train.sh` 와 동일 (`--model`, `--dataset`, `--stage2-mode`, `--stage1-mode`, `--stage1-epoch`).

**HF repo naming**
- base variant: `SaFD-00/{short}-{slug}base-stage2-{M2}-epoch{E2}`
- world-model variant: `SaFD-00/{short}-{slug}world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}` (예: `SaFD-00/qwen3-vl-8b-ac_2-world-model-stage1-full-epoch1-stage2-lora-epoch3`)

**로컬 산출**: `outputs/{DS}/merged/{MODEL}_stage2_{variant_key}/epoch-{E2}/`.

도움말: `bash scripts/stage2_merge.sh --help`.

### 7.1 AC

In [ ]:
!bash scripts/stage2_merge.sh --model qwen3-vl-8b --dataset AC --stage2-mode lora --stage1-mode full --stage1-epoch 1

### 7.2 AC_2

In [ ]:
!bash scripts/stage2_merge.sh --model qwen3-vl-8b --dataset AC_2 --stage2-mode lora --stage1-mode full --stage1-epoch 1

### 다른 변형 실행

- **다른 모델 / mode / 상류 계보**: Section 6 와 동일한 인자 교체 규칙.

### 8. Stage 2 Evaluation Pipeline

**흐름: `train → merge → eval`.** eval 은 HF Hub 에 push 된 Stage 2 merged repo 만 pull 하며, 로컬 `adapter/` · `merged/` 디렉토리에 의존하지 않는다. 학습 DS (`--train-dataset`, `AC | AC_2`) 와 평가 DS (`--eval-datasets`, `AC | AC_2 | MB`) 를 분리한다 — MC 는 Stage 2 데이터가 없어 학습 DS 로 거절된다.

**EVAL_DS 별 분기**:
- **AC** — ID + OOD 두 test 파일 (`gui-model_stage2_test_{id,ood}.jsonl`) 을 함께 추론 → `_action_eval.py score --test-id ... --pred-id ... --test-ood ... --pred-ood ...` 가 `action_metrics.json` 에 `overall` / `in_domain` / `out_of_domain` 3 섹션 기록.
- **AC_2** — 단일 파일 `gui-model_stage2_test.jsonl` 1 회 추론 → single-pair `overall` 1 섹션.
- **MB** — 단일 파일 `gui-model_stage2.jsonl` 1 회 추론 → single-pair `overall` 1 섹션.

**Variants**: `base` (zero-shot) · `{full|lora}_base` · `{full|lora}_world_model`. world-model variant 는 `--stage1-mode {full|lora} --stage1-epoch N` 으로 상류 Stage 1 epoch 의 HF 레포 계보 번호를 주입한다.

**HF naming**:
- base: `SaFD-00/{short}-{slug}base-stage2-{M2}-epoch{E2}`
- world-model: `SaFD-00/{short}-{slug}world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}`

**산출 경로**: `outputs/{TRAIN_DS}/eval/{MODEL}/stage2_eval/{variant}[/epoch-{E2}]/on-{EVAL_DS}/action_metrics.json`. 재실행 시 marker 존재 unit 은 variant × EVAL_DS 조합 별로 독립 skip.

> **메트릭 정의** (Step Accuracy): `correct = parse_ok ∧ type==gt.type ∧ field_match(type)`. field_match: `navigate_back` / `finish` 항상 통과, `click` / `long_click` 의 `index`, `scroll.direction`, `open_app.params.app`, `input.params.text` 일치. 정본은 `scripts/_action_eval.py`, 회귀 테스트 `tests/test_action_eval.py` (48 케이스).


### `scripts/stage2_eval.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--train-dataset` | (필수) | `AC` · `AC_2` 만 — MC · AC_3 미지원 |
| `--eval-datasets` | train DS | 콤마 구분 — `AC,AC_2,MB` |
| `--model` | `all` | 평가 대상 모델 short name |
| `--variants` | `base,full_base,lora_base,full_world_model,lora_world_model` | base/world-model × full/lora 매트릭스 |
| `--stage1-mode` | `full` | world-model variant 시 상류 Stage 1 mode |
| `--stage1-epoch` | `3` | world-model variant 시 상류 Stage 1 epoch (HF repo slug 의 `epoch{E1}`) |
| `--epochs` | `1,2,3` | Stage 2 merged repo epoch sweep |

채점기는 `_action_eval.py` (Stage 2 Action Prediction). EVAL_DS=`AC` 는 ID + OOD 두 jsonl 을 함께 추론하여 `action_metrics.json` 의 `overall` / `in_domain` / `out_of_domain` 3 섹션 기록, `AC_2` · `MB` 는 단일 파일 → `overall` 1 섹션.

**산출물**: `outputs/{TRAIN_DS}/eval/{MODEL}/stage2_eval/{variant}[/epoch-{E2}]/on-{EVAL_DS}/action_metrics.json`.

도움말: `bash scripts/stage2_eval.sh --help`. 아래 cell 들은 본 노트북이 정의하는 baseline + variant sweep · plotting 코드다.

In [ ]:
## Stage 2 Baseline — base model zero-shot 에 대해 in-distribution (AC: ID+OOD) + cross-benchmark (MB: overall) 평가.
## EVAL_DS=AC 는 _action_eval.py 가 overall/in_domain/out_of_domain 3-section 으로 기록.
## EVAL_DS=MB 는 단일 파일이므로 overall 1-section 만 기록 (single-pair 모드).
##
## 산출: outputs/AC/eval/{MODEL}/stage2_eval/base/on-{EVAL_DS}/action_metrics.json
!bash scripts/stage2_eval.sh --train-dataset AC --eval-datasets AC,MB --variants base


In [ ]:
## Stage 2 base variant HF Hub sweep — SaFD-00/...base-stage2-{M2}-epoch{E2}.
## 매 epoch 별로 in-distribution + cross-benchmark 평가.
##
## 산출: outputs/AC/eval/{MODEL}/stage2_eval/{full|lora}_base/epoch-{E2}/on-{EVAL_DS}/action_metrics.json
!bash scripts/stage2_eval.sh --train-dataset AC --eval-datasets AC,MB \
    --variants full_base,lora_base --epochs 1,2,3


In [ ]:
## Stage 2 world-model variant HF Hub sweep — Stage 1 계보 포함:
##   SaFD-00/...world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}
##
## --stage1-mode / --stage1-epoch 로 상류 Stage 1 merged repo 의 계보 번호를 지정한다.
## 아래 예시는 Stage 1 FULL epoch 3 계보 + Stage 2 full/lora 모두 sweep.
!bash scripts/stage2_eval.sh --train-dataset AC --eval-datasets AC,MB \
    --stage1-mode full --stage1-epoch 3 \
    --variants full_world_model,lora_world_model --epochs 1,2,3

## Stage 1 LORA 계보도 필요하면 주석 해제:
# !bash scripts/stage2_eval.sh --train-dataset AC --eval-datasets AC,MB \
#     --stage1-mode lora --stage1-epoch 3 \
#     --variants full_world_model,lora_world_model --epochs 1,2,3


In [ ]:
# NOTE: 이 셀은 scripts/_action_eval.py 와 글자 단위 동치를 유지하는 정본 복제다.
# 실제 평가는 ``python scripts/_action_eval.py score ...`` 또는 stage2_eval.sh 를
# 통해 실행한다. 이 셀은 로직 레퍼런스 / 빠른 디버깅용.

import json
import re
import sys
from collections import defaultdict

# Stage 2 Action Prediction evaluator (정본).
# scripts/_action_eval.py 와 글자 단위 동치를 유지한다.
# 메트릭: Step Accuracy (SA) — bounds 영구 부재 데이터셋용 정의.
#   correct = (parse_ok ∧ type==gt ∧ field_match(type))
# field_match: navigate_back/finish 항상 통과 · click/long_click 의 index ·
#              scroll.direction · open_app.params.app · input.params.text

# ── Action Parsing ───────────────────────────────────────────────────────
def parse_action(text):
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    match = re.search(r'\{[^{}]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return None


# ── Field 추출 헬퍼 (top-level + nested params 모두 지원) ────────────────
def _pval(action, key):
    if action is None:
        return None
    if key in action:
        return action[key]
    return (action.get('params') or {}).get(key)


def _norm(s):
    return str(s if s is not None else '').strip().lower()


# ── Step Accuracy 채점 ───────────────────────────────────────────────────
# field_match(type) 가 정의된 type 만 step_correct 로 인정. 그 외 type 은 False.
_FIELD_MATCH_TYPES = {
    'navigate_back', 'finish', 'click', 'long_click',
    'scroll', 'open_app', 'input',
}


def evaluate_single(gt_action, pred_action):
    result = {
        'parsed': pred_action is not None,
        'type_correct': False,
        'step_correct': False,
        'has_index_check': False,    # click / long_click
        'has_dir_check': False,      # scroll
        'has_app_check': False,      # open_app
        'has_text_check': False,     # input
    }
    if pred_action is None:
        return result

    gt_type = str(gt_action.get('type', '')).lower()
    pred_type = str(pred_action.get('type', '')).lower()
    result['type_correct'] = (gt_type == pred_type)
    if not result['type_correct']:
        return result

    # field_match
    if gt_type in ('navigate_back', 'finish'):
        result['step_correct'] = True
        return result

    if gt_type in ('click', 'long_click'):
        result['has_index_check'] = True
        result['step_correct'] = (
            str(gt_action.get('index')) == str(pred_action.get('index'))
        )
        return result

    if gt_type == 'scroll':
        result['has_dir_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'direction')) == _norm(_pval(pred_action, 'direction'))
        )
        return result

    if gt_type == 'open_app':
        result['has_app_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'app')) == _norm(_pval(pred_action, 'app'))
        )
        return result

    if gt_type == 'input':
        result['has_text_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'text')) == _norm(_pval(pred_action, 'text'))
        )
        return result

    # unknown type — type 만 일치해도 step 은 False (정책)
    return result


def evaluate_predictions(test_path, pred_path):
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    if len(gt_entries) != len(pred_entries):
        print(f"[warn] length mismatch: gt={len(gt_entries)} pred={len(pred_entries)} "
              f"→ truncating to {min(len(gt_entries), len(pred_entries))}", file=sys.stderr)

    results = []
    per_type = defaultdict(lambda: {
        'count': 0, 'type_correct': 0, 'step_correct': 0,
    })
    cond = {
        'index': {'n': 0, 'k': 0},   # click + long_click
        'dir':   {'n': 0, 'k': 0},   # scroll
        'app':   {'n': 0, 'k': 0},   # open_app
        'text':  {'n': 0, 'k': 0},   # input
    }

    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_action = json.loads(gt_entry['messages'][-1]['value'])
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))
        pred_action = parse_action(pred_text)

        r = evaluate_single(gt_action, pred_action)
        gt_type = str(gt_action.get('type', 'unknown')).lower()
        r['gt_type'] = gt_type
        results.append(r)

        per_type[gt_type]['count'] += 1
        per_type[gt_type]['type_correct'] += int(r['type_correct'])
        per_type[gt_type]['step_correct'] += int(r['step_correct'])

        if r['has_index_check']:
            cond['index']['n'] += 1
            cond['index']['k'] += int(r['step_correct'])
        if r['has_dir_check']:
            cond['dir']['n'] += 1
            cond['dir']['k'] += int(r['step_correct'])
        if r['has_app_check']:
            cond['app']['n'] += 1
            cond['app']['k'] += int(r['step_correct'])
        if r['has_text_check']:
            cond['text']['n'] += 1
            cond['text']['k'] += int(r['step_correct'])

    total = len(results)
    parsed = sum(1 for r in results if r['parsed'])
    type_correct = sum(int(r['type_correct']) for r in results)
    step_correct = sum(int(r['step_correct']) for r in results)

    parse_rate = parsed / total if total else 0
    type_acc = type_correct / total if total else 0
    step_acc = step_correct / total if total else 0

    per_type_summary = {}
    for t, d in per_type.items():
        per_type_summary[t] = {
            'count':    d['count'],
            'type_acc': round(d['type_correct'] / d['count'] if d['count'] else 0, 4),
            'step_acc': round(d['step_correct'] / d['count'] if d['count'] else 0, 4),
        }

    macro_step = (sum(v['step_acc'] for v in per_type_summary.values()) /
                  len(per_type_summary)) if per_type_summary else 0

    def _ratio(c):
        return c['k'] / c['n'] if c['n'] else 0

    return {
        'total':                total,
        'parse_rate':           round(parse_rate, 4),
        'type_accuracy':        round(type_acc, 4),
        'step_accuracy':        round(step_acc, 4),
        'macro_step_accuracy':  round(macro_step, 4),
        'cond_index_acc':       round(_ratio(cond['index']), 4),
        'cond_dir_acc':         round(_ratio(cond['dir']),   4),
        'cond_app_acc':         round(_ratio(cond['app']),   4),
        'cond_text_acc':        round(_ratio(cond['text']),  4),
        'per_type':             per_type_summary,
    }

print("[Loaded] Step Accuracy evaluator: parse_action, evaluate_single, evaluate_predictions")
